In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import openpyxl
import re
import os
from variableUtils import *
from Utils import *
from ClassUtils import *
from pprint import pprint
import json
from collections import defaultdict
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from reportlab.lib.pagesizes import letter, landscape, A4, A3
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, PageBreak, Paragraph, Spacer, Image, HRFlowable
from reportlab.lib import colors
from matplotlib.backends.backend_pdf import PdfPages
from reportlab.lib.enums import TA_CENTER, TA_RIGHT, TA_LEFT, TA_JUSTIFY
from reportlab.platypus import Paragraph, Spacer, KeepTogether, KeepInFrame 
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
import io
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import FormulaRule
import PIL
import ast
from adjustText import adjust_text
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from scipy.stats import f_oneway
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(sns.__version__)
from collections import Counter
from datetime import datetime
# get today's date
today = datetime.now().strftime('%d-%m-%Y')
print(f"Today's date: {today}")
plt.ioff()

0.13.2
Today's date: 09-12-2025


In [41]:
# Load the ebelWeights and ebelMatrix from the saved Excel files
ebelWeightsSavePath = os.path.join('2025', 'ebelWeights.xlsx')
ebelMatrixSavePath = os.path.join('2025', 'ebelMatrix.xlsx')
ebelWeights = pd.read_excel(ebelWeightsSavePath, sheet_name=None)
ebelMatrix = pd.read_excel(ebelMatrixSavePath, sheet_name=None)
importanceWeights = {"Essential": 5, "Important": 3, "Non-essential": 1}
def getWeight(row, ebelDf):
    importance = row['Importance']
    difficulty = row['Difficulty']
    return ebelDf.loc[importance, difficulty]


for key, df in ebelWeights.items():
    # if key!= 'BOH3-Examiner-2':
    #     continue

    df.set_index('MarkingChecklist', inplace=True)
    if 'scoreWeight' not in df.columns:
        df['scoreWeight'] = df['Importance'].map(importanceWeights)

    # ebel cutoffs
    if key not in ebelMatrix.keys():
        print(f"Key {key} not found in ebelMatrix")
        continue
    ebelDf = ebelMatrix[key].set_index('Importance')
    df['ebelWeight'] = df.apply(getWeight, axis=1, ebelDf=ebelMatrix[key].set_index('Importance'))
    ebelSum = df['ebelWeight'].sum()
    ebelCutoff = ebelSum /len(df)
    ebelWeightedSum = df['ebelWeight'] * df['scoreWeight']
    ebelWeightedSum = ebelWeightedSum.sum()
    ebelWeightedCutoff = ebelWeightedSum / df['scoreWeight'].sum()
    # if key!= '524':
        # continue
    print(f"========================\nKey: {key}")
    # display(df.head())
    print(f"EBEL Cutoff: {round(ebelCutoff,2)}")
    # print(f"EBEL Weighted Sum: {ebelWeightedSum.sum()}| {df['scoreWeight'].sum()}")
    print(f"EBEL Weighted Cutoff: {round(ebelWeightedCutoff,2)}")

Key: 521
EBEL Cutoff: 74.29
EBEL Weighted Cutoff: 74.5
Key: 522
EBEL Cutoff: 74.41
EBEL Weighted Cutoff: 74.59
Key: 523
EBEL Cutoff: 74.41
EBEL Weighted Cutoff: 74.59
Key: 524
EBEL Cutoff: 74.41
EBEL Weighted Cutoff: 74.59
Key: 532
EBEL Cutoff: 62.65
EBEL Weighted Cutoff: 65.0
Key: 533
EBEL Cutoff: 52.94
EBEL Weighted Cutoff: 54.06
Key: 534
EBEL Cutoff: 52.94
EBEL Weighted Cutoff: 54.06
Key: 577
EBEL Cutoff: 30.0
EBEL Weighted Cutoff: 30.0
Key: 577-2
EBEL Cutoff: 30.0
EBEL Weighted Cutoff: 30.0
Key: 578
EBEL Cutoff: 41.0
EBEL Weighted Cutoff: 42.37
Key: 531
EBEL Cutoff: 77.5
EBEL Weighted Cutoff: 79.91
Key: IC-BOH
EBEL Cutoff: 84.44
EBEL Weighted Cutoff: 86.97
Key: DDI-DDS
EBEL Cutoff: 70.0
EBEL Weighted Cutoff: 95.68
Key: SOR-DDS
EBEL Cutoff: 50.0
EBEL Weighted Cutoff: 91.53
Key: SOP-DDS
EBEL Cutoff: 87.5
EBEL Weighted Cutoff: 96.08
Key: MAR-31-DDS
EBEL Cutoff: 49.62
EBEL Weighted Cutoff: 50.55
Key: DDS2-MAR-31
EBEL Cutoff: 49.62
EBEL Weighted Cutoff: 50.55
Key: FS-DDS
EBEL Cutoff: 10

## Json to excel

In [42]:
jsonfilepath = '2025\\DDS2\dds2_v2.json'
# jsonfilepath = '2025/BOH1/boh1_v2.json'
# jsonfilepath = '2025/DDS1/dds1_v2.json'
# jsonfilepath = '2025\\BOH2\\boh2_v2.json'
# jsonfilepath = '2025\\Viva\\DDS3\\dds3.json'
# jsonfilepath = '2025\\Viva\\BOH2\\boh2.json'
# jsonfilepath = '2025\\OSCE\\DDS2\\dds2.json'
jsonfilepath = '2025\\OSCE\\BOH1\\boh1.json'
# jsonfilepath = '2025\\OSCE\\BOH2\\boh2.json'
# jsonfilepath = '2025\\mini CEX\\DDS2\\dds2_v2.json'
# jsonfilepath = '2025\\Extra\\Oral Pres\\data.json'
folder, file, ext = getFolderandFileName(jsonfilepath)
with open(jsonfilepath, 'r', encoding='utf-8') as file:
    data = json.load(file)
print(json.dumps(data, indent=4, ensure_ascii=False))

{
    "success": true,
    "result": [
        {
            "osce_id": 1437,
            "student_name": "Aarabhy Varathan",
            "student_email": "varathana@student.unimelb.edu.au",
            "student_number": 1699276,
            "assessor_name": "Clare McNally",
            "assessor_email": "mcnallyc@unimelb.edu.au",
            "date": "2025-11-04T00:00:00",
            "cohort": "BOH1",
            "subject": "ORAL10005",
            "station": 1,
            "forms": {
                "data": {
                    "assessor": {}
                },
                "scales": {
                    "scale-time-mgmt": {
                        "name": "Time Management Scale",
                        "fields": {
                            "1": "Lvl 1: Work not completed in allocated timeframe",
                            "2": "Lvl 2: Completes task in allocated timeframe"
                        },
                        "description": "Please use your professional judgem

In [43]:
# Fields to pull out from supervisor_data
rubricFields = ['time_mgmt', 'entrustment', 'communication', 'professionalism', 'patient_complexity']
supervisorMainFields = ['reflection', 'clinical_incident', 'comments', 'scale-section-rating', 'scale-practice-readiness', 'assessor_signature','scale-global-rating']
# supervisorMainFields += rubricFields
osce_fields = ["scale-time-mgmt","scale-global-rating", "scale-practice-readiness","scale-communication-clarity-terminology", "scale-communication-engagement-understanding",
               "scale-communication", "scale-entrustment"]
supervisorMainFields +=osce_fields
# Field to pull out from student_data
studentMainFields = ['reflection']
def getStudentList(listpath, cohort=None):
    listdf = pd.read_excel(listpath, keep_default_na=False, na_values=[''])
    if cohort is not None:
        listdf = listdf[listdf['Cohort'] == cohort]
    validIds = listdf['Student ID'].to_list()
    # convert to int
    
    validIds = [int(id) for id in validIds]
    print(f"Number of valid student IDs in {cohort}: {len(validIds)}")
    return validIds
validIds = getStudentList('2025\RE_ Student List.xlsx', cohort='BOH1')

def processtoRow(form_data, form_name, baseData):
    informdata = form_data.get('data', {})
    if not informdata:
        row = {**baseData, 'form_name': form_name}
        print(f"No data found for form {form_name}")
        return row
    student_data = informdata.get('student', {})
    # studentfieldsextracted = {k: student_data.get(k) for k in supervisorMainFields}
    studentfieldsextracted = {k: v for k, v in student_data.items() if k in studentMainFields}
    # rename the reflection field to student_reflection
    if 'reflection' in studentfieldsextracted:
        studentfieldsextracted['student_feedback'] = studentfieldsextracted.pop('reflection')


    studentRemainingData = {k: v for k, v in student_data.items() if k not in studentMainFields}

    assessor_data = informdata.get('assessor', {})
    # print(form_data.keys())
    if 'clinic' in form_data.keys():
        # print('Clinic is present')
        if form_data['clinic'] == "Smile Squad":
            assessor_data = student_data
            # print(assessor_data)
            print(f'assessor_data keys: {assessor_data.keys()}')
            print(f'Form data keys: {form_data.keys()}')
            print(f'Student Data keys: {student_data.keys()}')
            if 'assessor_signature' in assessor_data.keys():
                print(f"Assessor signature found in Smile Squad data {assessor_data['assessor_signature']}")
                assessor_name = assessor_data.pop('assessor_signature')
                baseData['assessor_name'] = assessor_name
            # print(assessor_data)

    rubricFieldsExtracted= {k: assessor_data[k]['scale'] for k in assessor_data.keys() if k in rubricFields} 
    assessorfieldsextracted={k: assessor_data[k] for k in assessor_data.keys() if k in supervisorMainFields}
    assessorfieldsextracted |= rubricFieldsExtracted
    # print(assessorfieldsextracted)
    # if assessor data is a dictionary, then extract the values in the dict
    # assessorfieldsextracted = {k: v if not isinstance(v, dict) else v['scale'] for k, v in assessorfieldsextracted.items()}
    # assessorfieldsextracted = {k: assessor_data.get(k) for k in supervisorMainFields}
    # rename the reflection field to assessor_reflection
    if 'reflection' in assessorfieldsextracted:
        # print("Renaming reflection to assessor_feedback")
        assessorfieldsextracted['assessor_feedback'] = assessorfieldsextracted.pop('reflection')
    if 'comments' in assessorfieldsextracted:
        # print("Renaming comments to assessor_feedback")
        assessorfieldsextracted['assessor_feedback'] = assessorfieldsextracted.pop('comments')
    # if the values in assess are None dict, then extract the values in the dict
    for k, v in assessorfieldsextracted.items():
        if isinstance(v, dict):
            # print(f"Extracting scale for {k} from assessor data")
            assessorfieldsextracted[k] = v['scale']
    assessorRemainingData = {k: v for k, v in assessor_data.items() if k not in supervisorMainFields + rubricFields}

    serializedFormData = {
        k: json.dumps(v) if isinstance(v, dict) else v
        for k, v in form_data.items()
    }
    serializedFormData['student_data'] = json.dumps(studentRemainingData) if isinstance(student_data, dict) else student_data
    serializedFormData['supervisor_data'] = json.dumps(assessorRemainingData) if isinstance(assessor_data, dict) else assessor_data
    
    # print(form_name, assessorfieldsextracted)
    
    row = {            
        'form_name': form_name,
        **baseData,
        **serializedFormData,
        **studentfieldsextracted,
        **assessorfieldsextracted,
    }
    return row

allEntries = []
print(data.keys())
for entry in data['result']:
    date = entry.get('datetime', 'Unknown Date')
    studentId = entry.get('student_number', 'Unknown ID')
    assessmentId = entry.get('assessment_id', 'Unknown Assessment ID')
    # if studentId not in validIds:
    #     print(f"Skipping entry for student ID {studentId} as it is not in the valid list.")
    #     continue
    baseData = {k: v for k, v in entry.items() if not isinstance(v, dict)}
    # display(pd.DataFrame([baseData]))

    # now the forms data
    forms = entry.get('forms', {})
    if "data" in forms:
        form_data = forms
        form_name = entry.get('assessment_id', entry.get('osce_id', entry.get('oral_presentation_id', 'Unknown Form ID')))
        row = processtoRow(form_data, form_name, baseData)
        allEntries.append(row)
    # if not forms:
    #     print(f"No forms data found for assessment ID {assessmentId} on {date}")
    #     row = {**baseData, 'form_name': np.nan}
    #     allEntries.append(row)
    #     continue
    else:
        for form_name, form_data in forms.items():
            row = processtoRow(form_data, form_name, baseData)
            # row['assessor_submitted'] = row.pop('submitted_by_assessor')
            # row['student_submitted'] = row.pop('submitted_by_student')
            allEntries.append(row)
            # print(f"Row Data: {row}")
        # break

df = pd.DataFrame(allEntries)
# drop the columns that are completely empty
df.dropna(axis=1, how='all', inplace=True)
# save to excel
df.to_excel(os.path.join(folder, f"all_data_combined_v2.xlsx"), index=False)
# check attendance
print(f"Number of unique student IDs: {df['student_number'].nunique()}")
# check for valid ids not in the df
missingIds = set(validIds) - set(df['student_number'].unique())
if missingIds:
    print(f"Missing student IDs not in the data: {missingIds}")  



Number of valid student IDs in BOH1: 55
dict_keys(['success', 'result', 'count'])
Number of unique student IDs: 58
Missing student IDs not in the data: {1734468, 1613866, 1760563, 1958038, 1032153, 1745117}


## Get scores

## OSCE

In [46]:
# workbookpath = '2025\\OSCE\\DDS2\\all_data_combined_v2.xlsx'
# workbookpath = '2025\\OSCE\\BOH2\\all_data_combined_v2.xlsx'
workbookpath = '2025\\OSCE\\BOH1\\all_data_combined_v2.xlsx'
folder, file, ext = getFolderandFileName(workbookpath)
df = pd.read_excel(workbookpath)
colId = 'Student ID'
colName = 'Student Name'
colDate = 'Date'
colCI = 'CI'
colTS = 'TS'
colES = 'ES'
colCS = 'CS'
colPS = 'PS'
colPractice = 'Practice Readiness'
colTM = 'Time Mgmt'
colGlobal = 'Global Rating'
colClarity = 'Clarity'
colEngagement = 'Engagement'
colCompleted = 'Completed'
colChecklists = 'checklists'
replaceNotReviwedwithNo = False
# display(df.describe(include='all'))


colAssessorName = 'Assessor Name'
colAssessorFeedback = 'Assessor Feedback'
colStation = 'Station'
df.rename(columns={'student_number':colId, 'student_name': colName, 'date': colDate, 'station': colStation,
                   'time_mgmt': colTS, 'entrustment': colES, 'communication': colCS, 'professionalism': colPS, 
                 'scale-practice-readiness': colPractice, 'scale-communication-clarity-terminology': colClarity, 
                   'clinical_incident': colCI, 'assessor_name': colAssessorName, 'scale-global-rating': colGlobal,
                   'assessor_feedback': colAssessorFeedback, 'scale-time-mgmt': colTM, 'scale-communication-engagement-understanding': colEngagement, 
                   'completed': colCompleted
                }, inplace=True, errors='ignore')
df[colId] = df[colId].astype(str)


# add rotation to as a column
colRotation = 'Rotation'
df[colDate] = pd.to_datetime(df[colDate], errors='coerce')
target_date = datetime(2025, 12, 9).date()
df = df[df[colDate].dt.date == target_date]

# rotationfile = '2025\OSCE\DDS2\Rotation Info with StudentIDs.csv'
# rotationdf = pd.read_csv(rotationfile)
# rotationdf[colId] = rotationdf[colId].astype(str)
# df = df.merge(rotationdf[[colId, colRotation]], on=colId, how='left')

# Convert relevant columns to Int64
df[colPractice] = df[colPractice].astype('Int64')
# df[colTM] = df[colTM].astype('Int64')
df[colGlobal] = df[colGlobal].astype('Int64')
# df[colClarity] = df[colClarity].astype('Int64')
# df[colEngagement] = df[colEngagement].astype('Int64')
df[colStation] = df[colStation].astype('Int64')
df = df[df[colCompleted] != False]
# subtract 1 from clarity engagement and practice
# df[colClarity] -= 1
# df[colEngagement] -= 1
# df[colPractice] -= 1

# define weights for practice, tm, clarity and engagement
rubricWeights = {
    4:{
        colClarity:0.0625,
        colEngagement:0.0625
    },
    5:{
        colEngagement:0.1,
        colClarity:0.1
    },
    12:{
        colEngagement:1.0/17,
        colClarity:1.0/17,
    }
}
rubricDenominators = {
    colTM: 1.0,
    colPractice: 3.0,
    colClarity: 3.0,
    colEngagement: 3.0
}
levelScores = {
    'Excellent':3,
    'Good':2,
    'Pass':1,
    'Fail':0,
    'Yes':3,
    'No':0
}
# levelScores = {'Yes'    :1,
#                'No'     :0}
levelScores2 = {
    'Very Good': 3,
    'Satisfactory': 2,
    'Borderline': 1,
    'Unsatisfactory': 0
}
modifiedLevelScores ={
    26: {'MC1': levelScores2, 'MC2': levelScores2, 'MC3': levelScores2, 'MC4': levelScores2, 'MC5': levelScores2}
}
pprint(rubricWeights)

# anonymize ids
allids = df[colId].unique()
# shuffle ids
np.random.shuffle(allids)
id_map = {id_: f"student_{i}" for i, id_ in enumerate(allids)}
colIdanon = 'Anonymous ID'
df[colIdanon] = df[colId].map(id_map)
# get counts of ids
id_counts = df[colId].value_counts()
print(id_counts)

display(df.head(5))


{4: {'Clarity': 0.0625, 'Engagement': 0.0625},
 5: {'Clarity': 0.1, 'Engagement': 0.1},
 12: {'Clarity': 0.058823529411764705, 'Engagement': 0.058823529411764705}}
Student ID
1758644    5
1758028    5
1766318    5
1475117    5
1764857    5
1746975    5
Name: count, dtype: int64


,form_name,osce_id,Student Name,student_email,Student ID,Assessor Name,assessor_email,Date,cohort,subject,Station,Completed,data,scales,checklists,student_data,supervisor_data,Global Rating,Practice Readiness,Assessor Feedback,Anonymous ID
527,2704,2704,Chloe Michael,cc.michael@student.unimelb.edu.au,1758644,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""1. Needs complete r...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Pass"", ""MC2""...",1,1,1. Needs complete revision. Not confident/Unsa...,student_2
528,2705,2705,Fujia Guo,fujiaguo@student.unimelb.edu.au,1758028,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""Identified teeth pr...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Excellent"", ...",3,2,Identified teeth present well.\nDetects majori...,student_1
529,2706,2706,Imijen Ellis,iaellis@student.unimelb.edu.au,1766318,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""Identified teeth ve...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Excellent"", ...",2,3,Identified teeth very well - but needed prompt...,student_5
530,2707,2707,Jackie Tran,jackietran@student.unimelb.edu.au,1475117,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""1. Missed carious l...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Excellent"", ...",2,2,1. Missed carious lesions or incorrectly inter...,student_3
531,2708,2708,Joyce Ye,kaixin.ye.2@student.unimelb.edu.au,1764857,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""Not quite there yet...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Excellent"", ...",1,1,"Not quite there yet, able to identify radioluc...",student_0


In [ ]:
# for 14th of October
df[colDate] = pd.to_datetime(df[colDate]).dt.date
dfoct = df[df[colDate] == datetime(2025, 10, 14).date()]
# filter out completed false
dfoct = dfoct[dfoct[colCompleted] != False]
# remove

def dropExtraConfigDeep(obj):
    """Recursively remove any key named 'extra_config' (case-insensitive)."""
    if isinstance(obj, dict):
        # delete matching keys first
        for k in list(obj.keys()):
            if k.lower() == 'extra_config':
                obj.pop(k, None)
        # recurse into remaining values
        for k, v in list(obj.items()):
            obj[k] = dropExtraConfigDeep(v)
        return obj
    if isinstance(obj, list):
        return [dropExtraConfigDeep(v) for v in obj]
    return obj

def cleanCell(cell):
    """Parse JSON string → remove extra_config → return same Python type."""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return cell
    data = json.loads(cell) if isinstance(cell, str) else cell
    return dropExtraConfigDeep(data)

# apply function
df['checklists'] = df['checklists'].apply(cleanCell)
df['checklists'] = df['checklists'].apply(lambda x: json.dumps(x))
display(dfoct.head())


### Score processing

In [47]:
def extractCodes(supervisorDataStr):
    try:
        data = json.loads(supervisorDataStr)
        codes = data.keys()
        return sorted(codes)
    except Exception as e:
        print(f"Error extracting codes: {e}")
        return []
    
# Expand the JSON with scores having two levels of keys
def expandJson(row):
    jsonDict = json.loads(row)
    flatDict = {}
    for outerKey, innerDict in jsonDict.items():
        for innerKey, value in innerDict.items():
            flatDict[f'{outerKey}_{innerKey}'] = value
    return pd.Series(flatDict)

# get row wise scores for each item code
def getRowWiseScores(df):
    df['supervisor_data'] = df['supervisor_data'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    # df.replace({'Complete': 'Yes', 'Incomplete': 'No', 'Not complete': 'No', 'Not Complete': 'No'}, inplace=True)
    scoresList = []
    
    for idx, row in df.iterrows():    
        print(f"Row: {idx}")
        station = row[colStation]
        jsonData = row['supervisor_data']
        if not isinstance(jsonData, dict):
            print(f"Row {idx} has no supervisor_data or it is not a dictionary\n{jsonData}")
            scoresList.append(json.dumps({}))
            continue
        scoreDict = {}
        # print(f'Item Codes: {itemList}')
        for item, mcDict in jsonData.items():
            # print(f'Item Code: {item}')
            # replace Not Reviewed and Not Assessed with No in mcDict values
            mcDict = {k: ('NA' if v in ['Not Reviewed', 'Not Assessed'] else v) for k, v in mcDict.items()}
            if replaceNotReviwedwithNo:
                mcDict = {k: ('No' if v =='NA' else v) for k, v in mcDict.items()}

            scoreDict[item] = {}
            cutoff = 0
            weightedCutoff = 0
            weightedSum = 0
            denominator = 0
            nYes = 0
            nNo = 0
            # print(mcDict)
            notNADict = {k: v for k, v in mcDict.items() if v != 'NA'}
            nNA  = len(mcDict) - len(notNADict)
            for key, value in notNADict.items(): # key is MC1, MC2 etc and value is Yes/No
                if value == 'Yes' or value == 'Complete':
                    nYes += 1
                elif value == 'No' or value == 'Incomplete' or value == 'Not Complete' or value == 'Not complete':
                    nNo += 1
                try:
                    cutoff += ebelWeights[item].loc[key, 'ebelWeight']
                    weightedCutoff += ebelWeights[item].loc[key, 'ebelWeight'] * ebelWeights[item].loc[key, 'scoreWeight']
                    denominator += ebelWeights[item].loc[key, 'scoreWeight']
                    if value == 'Yes' or value == 'Complete':
                        weightedSum += ebelWeights[item].loc[key, 'scoreWeight']
                    elif value == 'No' or value == 'Incomplete' or value == 'Not Complete' or value == 'Not complete':
                        weightedSum += 0
                except KeyError as e:
                    print(f"KeyError: {item} {key} not found in ebelWeights for row {idx}\n{e}")
                    continue
            
            scoreDict[item]['Yes'] = nYes
            scoreDict[item]['No'] = nNo
            scoreDict[item]['NA'] = nNA
            scoreDict[item]['% Yes'] = round((nYes / (nYes + nNo) * 100), 2) if (nYes + nNo) > 0 else np.nan
            # print(scoreDict[item])
            if not notNADict:
                print(f'Item Code: {item} has no data')
                scoreDict[item] = scoreDict[item] | {'Cutoff': np.nan, 'Weighted Cutoff': np.nan, 'Weighted Sum': np.nan, 'Denominator': np.nan,
                                   'Weighted Score': np.nan, 'Total Score': np.nan}
                continue
            
            try:
                cutoff = cutoff / len(notNADict)
            except ZeroDivisionError as e:
                print(f'ZeroDivisionError for item {item} with data: {notNADict}\n{e}')
                cutoff = np.nan
            
            try:
                weightedCutoff = weightedCutoff / denominator
                weightedScore = round((weightedSum / denominator * 100), 2)
            except ZeroDivisionError as e:
                print(f'ZeroDivisionError for item {item} with data: {notNADict}\n{e}')
                weightedCutoff = np.nan
                weightedScore = np.nan

            totalScore = 0
            weightRubric = 0
            totalCutoff = 0
            if station not in rubricWeights.keys():
                totalScore = weightedScore
                totalCutoff = weightedCutoff
            else:    
                for rubric, weight in rubricWeights[station].items():
                    # print(row[key], type(row[key]), key)
                    if pd.notna(row[rubric]):
                        totalScore += weight*(row[rubric]/rubricDenominators[rubric])*100
                        weightRubric += weight
                        addValue = weight* (0.66)*100
                        totalCutoff += addValue # 66% of rubric is pass
                weightMC = 1 - weightRubric
                totalScore += weightMC * weightedScore
                totalCutoff += weightMC * weightedCutoff

            # print(f'Cutoff: {cutoff}, Weighted Cutoff: {weightedCutoff}, Weighted Sum: {weightedSum}, Denominator: {denominator}')
            try:
                scoreDict[item] = scoreDict[item] | {'Cutoff': round(cutoff,2), 'Weighted Cutoff': round(weightedCutoff,2), 'Weighted Sum': int(weightedSum), 'Denominator': int(denominator),
                                                    'Weighted Score': weightedScore, 'Total Score': round(totalScore, 0), 'Total Cutoff': round(totalCutoff, 0)}
            except TypeError:
                print(f"Error in item {item} with data: {scoreDict[item]}, student data: {row[colId]}")
                print(cutoff, weightedCutoff, weightedSum, denominator, weightedScore, totalScore)
        # pprint(scoreDict)        
        scoresList.append(json.dumps(scoreDict))
    df['Scores'] = scoresList
    df['supervisor_data'] = df['supervisor_data'].apply(lambda x: json.dumps(x, indent=2, ensure_ascii=False))

def getScoreLevels(df):
    df['supervisor_data'] = df['supervisor_data'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    df[colChecklists] = df[colChecklists].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    # df.replace({'Complete': 'Yes', 'Incomplete': 'No', 'Not complete': 'No', 'Not Complete': 'No'}, inplace=True)
    scoresList = []

    for idx, row in df.iterrows():    
        print(f"Row: {idx}")
        station = row[colStation]
        checklistInfo = row[colChecklists]
        checklistKeys = list(checklistInfo.keys())
        checklistKey = checklistKeys[0]  # This will be something like "OHP1-Station-6"
        checklistData = checklistInfo[checklistKey]
        print(f"Checklist Data: {checklistData}")
        rubric = checklistData['extra_config']['rubric']

        jsonData = row['supervisor_data']
        if not isinstance(jsonData, dict):
            print(f"Row {idx} has no supervisor_data or it is not a dictionary\n{jsonData}")
            scoresList.append(json.dumps({}))
            continue
        scoreDict = {}
        # print(f'Item Codes: {itemList}')
        for item, mcDict in jsonData.items():
            valueCounts = Counter(mcDict.values())
            valueCounts = dict(valueCounts)
            fullCounts = {k: valueCounts.get(k, 0) for k in levelScores.keys()} # number of occurrences of each level
            # scoreDict[item] = dict(fullCounts)
            scoreDict[item] = {}
            totalScore = 0
            denominator = 0
            # for key, value in dict(valueCounts).items():
            #     if key in levelScores:
            #         totalScore += levelScores[key] * value
            #         denominator += value*levelScores['Very Good']
            
            for mckey, value in mcDict.items():
                print(rubric)
                options = rubric[mckey] # a dictionary of O1: 'Excellent', O2: 'Good' etc
                print(f"Options: {options}")
                optionvalues = options.values()
                print(f"Value: {value}, Option Values: {optionvalues}")
                # if value is Good/Yes and optionvalues are Excellent, Good, Borderline, Pass then replace Good/Yes with Good
                if value == 'Good / Yes':
                    if 'Good' in optionvalues:
                        value = 'Good'
                    elif 'Yes' in optionvalues:
                        value = 'Yes'
                if value == 'Fail / No':
                    if 'Fail' in optionvalues:
                        value = 'Fail'
                    elif 'No' in optionvalues:
                        value = 'No'
                if mckey in modifiedLevelScores.get(station, {}):
                    print(mckey, value, modifiedLevelScores[station][mckey])
                    totalScore += modifiedLevelScores[station][mckey][value]
                    denominator += 3
                else:
                    totalScore += levelScores[value]
                    denominator += max(levelScores.values())

            scoreDict[item]['Total MC'] = totalScore
            scoreDict[item]['Denominator'] = denominator
            scoreDict[item]['Weighted MC Score'] = round(totalScore / denominator * 100, 2) if denominator > 0 else 0
            
            # add colClarity and colEngagement if present
            # if colClarity in row and pd.notna(row[colClarity]):
            #     totalScore += row[colClarity] 
            #     denominator += 3
            # if colEngagement in row and pd.notna(row[colEngagement]):
            #     totalScore += row[colEngagement]
            #     denominator += 3
            
            # scoreDict[item]['Total with Rubric'] = totalScore
            # scoreDict[item]['Denominator with Rubric'] = denominator
            # totalScore = 0
            # weightRubric = 0
            # for key, value in rubricWeights.items():
            #     # print(row[key], type(row[key]), key)
            #     if pd.notna(row[key]):
            #         totalScore += (value*row.get(key, 0)/rubricDenominators[key])*100
            #         weightRubric += value
            # weightMC = 1 - weightRubric
            # totalScore += weightMC * scoreDict[item]['Weighted MC Score']
            scoreDict[item]['Total Score'] = round(totalScore/denominator*100, 0) if denominator > 0 else 0

        scoresList.append(json.dumps(scoreDict))
    df['Scores'] = scoresList
    df['supervisor_data'] = df['supervisor_data'].apply(lambda x: json.dumps(x, indent=2, ensure_ascii=False))

In [48]:
df_ = df.copy() 
display(df_.head(3))
# loop over stations
brStations = [2, 6, 10, 16] # stations we will use borderline regression other we will use ebel
df_['Item Codes'] = df_['supervisor_data'].apply(extractCodes)
stations = df_[colStation].unique()
stations = np.sort(stations)
filepath = f'{folder}//stationWiseScores.xlsx'
stationDfDict = {}
for station in stations:
    print(f"Processing station: {station}")
    stationData = df_[df_[colStation] == station]
    # Further processing for each station
    # if station not in brStations:
    if False:
        getRowWiseScores(stationData)
        # save the station data to excel
    else:
        getScoreLevels(stationData)
    if os.path.exists(filepath):
        with pd.ExcelWriter(filepath, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            stationData.to_excel(writer, sheet_name=f"station_{station}", index=False)
    else:
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            stationData.to_excel(writer, sheet_name=f"station_{station}", index=False)
    stationDfDict[station] = stationData
    print(f"Saved station {station} data with EBEL scores.")

,form_name,osce_id,Student Name,student_email,Student ID,Assessor Name,assessor_email,Date,cohort,subject,Station,Completed,data,scales,checklists,student_data,supervisor_data,Global Rating,Practice Readiness,Assessor Feedback,Anonymous ID
527,2704,2704,Chloe Michael,cc.michael@student.unimelb.edu.au,1758644,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""1. Needs complete r...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Pass"", ""MC2""...",1,1,1. Needs complete revision. Not confident/Unsa...,student_2
528,2705,2705,Fujia Guo,fujiaguo@student.unimelb.edu.au,1758028,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""Identified teeth pr...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Excellent"", ...",3,2,Identified teeth present well.\nDetects majori...,student_1
529,2706,2706,Imijen Ellis,iaellis@student.unimelb.edu.au,1766318,Quor-Ten Teh,quor.teh@unimelb.edu.au,2025-12-09,BOH1,ORAL10005,2,True,"{""assessor"": {""comments"": ""Identified teeth ve...","{""scale-global-rating"": {""name"": ""Global Ratin...","{""OHP1-Station-2-Resit"": {""name"": ""OHP1 OSCE S...",{},"{""OHP1-Station-2-Resit"": {""MC1"": ""Excellent"", ...",2,3,Identified teeth very well - but needed prompt...,student_5


Processing station: 1
Row: 551
Checklist Data: {'name': 'OHP1 OSCE Station 1', 'fields': {'MC1': 'Maintains correct ergonomic positioning of both operator and patient.', 'MC2': 'Selects universal curette for the task.', 'MC3': 'Maintains a modified pen grasp throughout the procedure.', 'MC4': 'Uses the correct working end for the surface being debrided.', 'MC5': 'Inserts the curette with the face as close as possible to the tooth surface (0–40°).', 'MC6': 'Opens the face to achieve a 70–80° working angulation, indicated by correct positioning of the terminal shank relative to the long axis of the tooth (tilt the terminal shank slightly toward the tooth surface)', 'MC7': 'Adapts the toe-third of the cutting edge to the tooth surface.', 'MC8': 'Uses continuous, overlapping, controlled strokes directed away from the base of the pocket.', 'MC9': 'Applies appropriate lateral pressure—firm enough for debridement but without loss of instrument control.', 'MC10': 'Uses a range of stroke direct

In [49]:
rubricCols = [colPractice, colTM, colClarity, colEngagement, colGlobal]
feedbackCols = [colAssessorFeedback]   
beforeCols = ['osce_id', colName, colId, colIdanon, colAssessorName, colStation, 'Item Codes', 'checklists']

In [50]:
def flattenJson(jsonData):
    flatDict = {}
    for outerKey, innerDict in jsonData.items():
        if isinstance(innerDict, dict):
            for innerKey, value in innerDict.items():
                flatDict[f'{outerKey}_{innerKey}'] = value
        else:
            flatDict[outerKey] = innerDict
    return flatDict

def getSeparatedItemDf(df, dftype_='by item'):
    dfDict = {}
    # check if item codes is a string and convert to list
    df['Item Codes'] = df['Item Codes'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    itemCodes = df['Item Codes'].sum()
    itemCodes = list(set(itemCodes))
    for item in itemCodes:
        # print(f'Item Code: {item}')
        # itemDf.set_index('assessment_id', inplace=True)
        checklist = df[colChecklists].iloc[0]
        checklistKeys = list(checklist.keys())
        checklistKey = checklistKeys[0]  # This will be something like "OHP1-Station-6"
        checklistData = checklist[checklistKey]
        # print(f"Checklist Data: {checklistData}")
        options = checklistData['extra_config']['rubric']
        # print(f"Options for item {item}: {options}")
        itemDf = df[df['Item Codes'].apply(lambda x: item in x)] # select rows where item is in Item Codes
        if isinstance(itemDf['supervisor_data'].iloc[0], str):
            # print("Converting supervisor_data to JSON")
            # convert to json
            itemDf['supervisor_data'] = itemDf['supervisor_data'].apply(lambda x: json.loads(x))
            itemDf['supervisor_data'] = itemDf['supervisor_data'].apply(flattenJson)

        # get option values for the item
        supervisorDataDf= itemDf['supervisor_data'].apply(pd.Series)
        if replaceNotReviwedwithNo:
            supervisorDataDf.replace({'Not Reviewed': 'No'}, inplace=True)
        supervisorDataDf.replace({'Not Reviewed': 'NA', 'Not Assessed': 'NA'}, inplace=True)
        # display(itemDf.head())
        # display(supervisorDataDf.head())
        # only take columns have item code in them
        # display(supervisorDataDf.head())
        validCols = [col for col in supervisorDataDf.columns if f'{item}_MC' in col]
        supervisorDataDf = supervisorDataDf[validCols]
        # rename columns to remove assessor_ text
        supervisorDataDf.columns = [col.split('assessor_')[-1] for col in supervisorDataDf.columns]
        for col in supervisorDataDf.columns:
            # get the option values
            mcOptions = options[col.split(f'{item}_')[-1]]
            print(f"Column: {col}, Options: {mcOptions}")
            if 'Yes' in mcOptions.values():
                print(f"Replacing Good / Yes and Fail / No in column {col}")
                supervisorDataDf[col] = supervisorDataDf[col].replace({'Good / Yes': 'Yes', 'Fail / No': 'No'})
            else:
                print(f"Replacing Good / Yes and Fail / No in column {col} with Good and Fail")
                supervisorDataDf[col] = supervisorDataDf[col].replace({'Good / Yes': 'Good', 'Fail / No': 'Fail'})
        # Step 1: Convert Yes/No to 1/0
        supervisordataDfBinary = supervisorDataDf.replace({'Yes': 1, 'No': 0, 'NA': np.nan}|levelScores|{'Complete': 1, 'Incomplete': 0, 'Not Complete': 0, 'Not complete': 0})
        # display(supervisordataDfBinary.head())
        # Step 2: Calculate column scores
        # columnScores = supervisordataDfBinary.sum(axis=0)

        # Step 3: Sort columns based on score (descending)
        # sortedColumns = columnScores.sort_values(ascending=False).index
        sortedColumns = supervisorDataDf.columns # unsort the columns
        # Step 4: Reorder dataframe columns
        sortedSupervisorDataDf = supervisorDataDf[sortedColumns]
        sortedSupervisorDataDf = sortedSupervisorDataDf.replace({'Yes': 1, 'No': 0}|levelScores|{'Complete': 1, 'Incomplete': 0, 'Not Complete': 0, 'Not complete': 0})
        # remove item_ from column names
        sortedSupervisorDataDf.columns = [col.split(f'{item}_')[-1] for col in sortedSupervisorDataDf.columns]
        # display(sortedSupervisorDataDf.head())
        # display(supervisorDataDf.head())
        thisbeforeCols = [col for col in beforeCols if col in itemDf.columns]
        thisrubricCols = [col for col in rubricCols if col in itemDf.columns]
        combinedDf= pd.concat([itemDf[thisbeforeCols], sortedSupervisorDataDf, itemDf[thisrubricCols]], axis=1)
        # display(combinedDf.head())
        expandedScores = itemDf['Scores'].apply(expandJson)
        validCols2 = [col for col in expandedScores.columns if item in col]
        expandedScores = expandedScores[validCols2]
        # rename columns to remove item_ text
        expandedScores.columns = [col.split(f'{item}_')[-1] for col in expandedScores.columns]
        # display(expandedScores.head())
        combinedDf = pd.concat([combinedDf, expandedScores], axis=1)
        combinedDf = pd.concat([combinedDf, itemDf[feedbackCols]], axis=1)
        # sort by Total Score
        combinedDf.sort_values(by=['Total Score'], ascending=False, inplace=True)
            # in the checklists there is mappinng of MC1, MC2 etc to the actual checklist item names, create a second header row with that mapping
        if 'checklists' in combinedDf.columns:
            def getChecklistMapping(checklistStr, item):
                data = json.loads(checklistStr) if isinstance(checklistStr, str) else checklistStr
                if item in data:
                    mcDict = data[item]
                    mapping = {k: v for k, v in mcDict.items() if k.startswith('MC')}
                    return mapping
                else:
                    return {}
        
            firstRow = combinedDf.iloc[0]
            checklistMapping = getChecklistMapping(firstRow['checklists'], item)
            # create a new header row
            newHeader = []
            for col in combinedDf.columns:
                if col in checklistMapping:
                    newHeader.append(checklistMapping[col])
                else:
                    newHeader.append(col)
            # create a new df with the new header as the first row
            newDf = pd.DataFrame(columns=newHeader)
            combinedDf.columns = newHeader
            combinedDf = pd.concat([newDf, combinedDf], ignore_index=True)
            combinedDf.drop(columns=['checklists'], inplace=True)
        dfDict[item] = combinedDf  
    # combine all dfDict df into one df and then return that
    combinedDf = pd.concat(dfDict.values(), axis=0, ignore_index=True)
    combinedDf.sort_values(by='Total Score', ascending=False, inplace=True)



    return combinedDf



In [51]:
filepath = f'{folder}\\BOH1 OSCE Scores ({today}).xlsx'
for station, stationData in stationDfDict.items():
    # Process each station's data
    print(f"Processing data for station: {station}")
    dfProcessedStation = getSeparatedItemDf(stationData, dftype_='by item')
    # dfProcessedStation.sort_values(by=[colId], ascending=False, inplace=True)
    # remove columns with all empty values
    dfProcessedStation.dropna(axis=1, how='all', inplace=True)
    if os.path.exists(filepath):
        with pd.ExcelWriter(filepath, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            dfProcessedStation.to_excel(writer, sheet_name=f'Station {station}', index=False)
    else:
        with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
            dfProcessedStation.to_excel(writer, sheet_name=f'Station {station}', index=False)

Processing data for station: 1
Column: OHP1-Station-1-Resit_MC1, Options: {'O1': 'Excellent', 'O2': 'Good', 'O3': 'Pass', 'O4': 'Fail'}
Replacing Good / Yes and Fail / No in column OHP1-Station-1-Resit_MC1 with Good and Fail
Column: OHP1-Station-1-Resit_MC2, Options: {'O1': '', 'O2': 'Yes', 'O3': '', 'O4': 'No'}
Replacing Good / Yes and Fail / No in column OHP1-Station-1-Resit_MC2
Column: OHP1-Station-1-Resit_MC3, Options: {'O1': 'Excellent', 'O2': 'Good', 'O3': 'Pass', 'O4': 'Fail'}
Replacing Good / Yes and Fail / No in column OHP1-Station-1-Resit_MC3 with Good and Fail
Column: OHP1-Station-1-Resit_MC4, Options: {'O1': '', 'O2': 'Yes', 'O3': '', 'O4': 'No'}
Replacing Good / Yes and Fail / No in column OHP1-Station-1-Resit_MC4
Column: OHP1-Station-1-Resit_MC5, Options: {'O1': 'Excellent', 'O2': 'Good', 'O3': 'Pass', 'O4': 'Fail'}
Replacing Good / Yes and Fail / No in column OHP1-Station-1-Resit_MC5 with Good and Fail
Column: OHP1-Station-1-Resit_MC6, Options: {'O1': 'Excellent', 'O2': 

### BR test

In [ ]:
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM
from scipy.stats import shapiro, levene, kruskal

# cleanworkbookpath = '2025\\OSCE\\DDS2\\DDS2 OSCE unweighted (17-09-2025).xlsx'
cleanworkbookpath = '2025\\OSCE\\BOH2\\BOH2 OSCE Scores (11-11-2025).xlsx'
cleanworkbookpath = '2025\\OSCE\\BOH2\\BOH1 OSCE Scores (09-12-2025).xlsx'
folder, file, ext = getFolderandFileName(cleanworkbookpath)
# iterate through the sheets in the workbook
# Read all sheets into dictionary of DataFrames
sheetsDict = pd.read_excel(cleanworkbookpath, sheet_name=None)
def cronbachAlpha(df):
    """
    Compute Cronbach's alpha for a dataframe of items (e.g., MC1–MCn).
    Each row = student, each column = checklist item.
    """
    df = df.dropna(axis=1, how='all').dropna(axis=0, how='any')
    k = df.shape[1]
    if k < 2:
        return np.nan

    itemVar = df.var(axis=0, ddof=1)
    totalVar = df.sum(axis=1).var(ddof=1)
    # print(itemVar, totalVar)
    alpha = (k / (k - 1)) * (1 - (itemVar.sum() / totalVar))
    return alpha

def assessorEffectTest(df, colAssessor='Assessor Name', colScore='Total Score', colFallbackScore='% Yes'):
    if colScore not in df.columns:
        if colFallbackScore not in df.columns:
            raise ValueError("Score column not found.")
        colScore = colFallbackScore

    df = df[[colAssessor, colScore]].dropna()
    df[colScore] = pd.to_numeric(df[colScore], errors='coerce')
    df = df.dropna()

    # Group by assessor
    grouped = df.groupby(colAssessor)
    groupScores = [group[colScore].dropna().values for _, group in grouped if len(group) > 1]

    if len(groupScores) < 2:
        return np.nan, np.nan, "Insufficient groups"

    # 1. Shapiro-Wilk normality test
    normalityDict = {}
    normalityViolated = False
    for name, group in grouped:
        scores = group[colScore].dropna()
        if len(scores) >= 3:
            W, p = shapiro(scores)
            normalityDict[name] = round(p, 4)
            if p < 0.05:
                normalityViolated = True
        else:
            normalityDict[name] = None  # Not enough data

    # 2. Levene's test for equal variances
    leveneStat, leveneP = levene(*groupScores)
    leveneViolated = leveneP < 0.05

    # 3. Decision: ANOVA or Kruskal
    if normalityViolated or leveneViolated:
        stat, p = kruskal(*groupScores)
        testType = "Kruskal-Wallis"
    else:
        stat, p = f_oneway(*groupScores)
        testType = "ANOVA"

    return stat, p, testType



def safeMixedLM(endog, exog, groups):
    try:
        model = MixedLM(endog, exog, groups=groups)
        result = model.fit()
        return result
    except np.linalg.LinAlgError as e:
        print("⚠️ LinAlgError: Singular matrix encountered.")
        print(f"Groups unique values: {np.unique(groups)} (n={len(np.unique(groups))})")
        print(f"endog variance: {np.var(endog):.4f}, exog variance: {np.var(exog, axis=0)}")
        print(f"Sample size: {len(endog)}")
        return None
    except Exception as e:
        print(f"⚠️ Other error in MixedLM: {e}")
        return None
def calculateAssessorVarianceComponent(df, colGlobal='Global Rating', colScore='Total Score', colAssessor='Assessor'):
    """
    Calculates variance components using mixed-effects model to estimate
    the proportion of variance in scores explained by assessor effects.

    Parameters:
    - df: pandas DataFrame with scores, global rating, and assessor columns
    - colGlobal: name of the global rating column (for fixed effect)
    - colScore: name of the score column (outcome variable)
    - colAssessor: name of the assessor column (random effect group)

    Returns:
    - assessorVar: variance component of assessor
    - residualVar: residual (student-level) variance
    - assessorPercent: % variance attributable to assessor
    - model.summary(): model fit summary
    """
    # Drop missing and coerce numeric
    df = df[[colGlobal, colScore, colAssessor]].dropna()
    df[colGlobal] = pd.to_numeric(df[colGlobal], errors='coerce')
    df[colScore] = pd.to_numeric(df[colScore], errors='coerce')
    df = df.dropna()

    # Prepare data for model
    endog = df[colScore]
    exog = sm.add_constant(df[[colGlobal]])  # fixed effect: global rating
    groups = df[colAssessor]                # random effect: assessor

    # Fit mixed model
    result = safeMixedLM(endog, exog, groups)
    if result is None:
        # Skip variance extraction, mark as failed
        assessorVar, residualVar, percentVar = np.nan, np.nan, np.nan
    else:
        assessorVar = result.cov_re.iloc[0, 0]
        residualVar = result.scale
        percentVar = assessorVar / (assessorVar + residualVar) * 100

    return assessorVar, residualVar, percentVar, result

def brAnalysis(df, colScore='Total Score', colFallbackScore='% Yes', passLevel=2, sheetName= '', savegraph = True):
    """
    Perform Borderline Regression analysis to evaluate OSCE quality.
    
    Parameters:
    - df: pandas DataFrame
    - colGlobal: column for global rating scale
    - colScore: primary column for checklist score
    - colFallbackScore: fallback column if colScore is missing
    - passLevel: numeric level representing a passing grade (usually 3)
    - sheetName: name for plot title
    """

    savefolder = f'{folder}/graphs unweighted'
    os.makedirs(savefolder, exist_ok=True)
    # Use fallback score if main score not present
    if colScore not in df.columns:
        if colFallbackScore not in df.columns:
            raise ValueError("Neither primary nor fallback score column found.")
        colScore = colFallbackScore

    # Drop NA and convert global rating to numeric
    # df = df[[colGlobal, colScore]].dropna()
    print(f'\n{sheetName} - {len(df)} records')
    print(f'Col Score: {colScore} | col global: {colGlobal}')
    df[colGlobal] = pd.to_numeric(df[colGlobal], errors='coerce')
    # df[colGlobal] = df[colGlobal] - 1
    # passLevel = passLevel - 1
    df[colScore] = pd.to_numeric(df[colScore], errors='coerce')
    # display(df[[colGlobal, colScore]].head())
    # df = df.dropna()

    # ========== BLR Regression ==========
    X = df[[colGlobal]]
    y = df[colScore]
    X_const = sm.add_constant(X) # adding a constant for intercept
    model = sm.OLS(y, X_const).fit()
    intercept, slope = model.params.const, model.params[colGlobal]
    r2 = model.rsquared
    adj_r2 = model.rsquared_adj
    interGradeDiscrimination = slope


    # === Normality of residuals ===
    normalityResults = {}
    for level, group in df.groupby(colGlobal):
        scores = group[colScore].dropna()
        if len(scores) >= 3:  # Shapiro needs at least 3 values
            stat, p = shapiro(scores)
            normalityResults[level] = {"n": len(scores), "stat": round(stat, 3), "p": round(p, 4)}
        else:
            normalityResults[level] = {"n": len(scores), "stat": None, "p": None}

    # === Homogeneity of Variances ===
    groupedScores = [group[colScore].dropna() for _, group in df.groupby(colGlobal) if len(group) > 1]
    leveneStat, leveneP = levene(*groupedScores)

    # === Kruskal-Wallis Test ===
    
    
    # ========== Optimal Cut Score (at given passLevel) ==========
    cutScore = intercept + slope * passLevel
    cutScore = round(cutScore, 0)

    # ========= Assessor Variation =================
    fStatass, pValass, testType = assessorEffectTest(df, colAssessor='Assessor Name')
    assessorVar, residualVar, assessorPercent, result = calculateAssessorVarianceComponent(
    df, colGlobal=colGlobal, colScore=colScore, colAssessor=colAssessorName
        )
    
    # # ========= Item code variation ===============
    # fStatItem, pValItem, testType = assessorEffectTest(df, colAssessor=colRotation)
    # itemCodeVar, residualVar, itemCodePercent, result = calculateAssessorVarianceComponent(
    # df, colGlobal=colGlobal, colScore=colScore, colAssessor=colRotation
    #     )

    # ========== Inter-Group Variation ==========
    df['GlobalGroup'] = df[colGlobal].astype(int)
    groupData = [group[colScore].values for _, group in df.groupby('GlobalGroup')]
    f_stat, p_value = f_oneway(*groupData)

    # ============= Cronbach's Alpha ============
    checklistCols = [col for col in df.columns if col.startswith("MC")]
    alpha = cronbachAlpha(df[checklistCols])

    # Failure Rate
    failureRate = (df[colScore] < cutScore).mean() * 100
    passRate = (df[colScore] >= cutScore).mean() * 100
    globalCoverage = df[colGlobal].agg(['min', 'max']).to_dict()
    globalSpread = df[colGlobal].nunique()

    # ========== Plot ==========
    plt.ioff()
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=colGlobal, y=colScore, alpha=0.6)
    x_vals = np.array([0, 5])
    y_vals = intercept + slope * x_vals
    plt.plot(x_vals, y_vals, color='red', linestyle='--', label='Regression Line')
    plt.axhline(cutScore, color='green', linestyle=':', label=f'Cut Score (Global={passLevel}): {cutScore:.2f}')
    plt.title(f'Borderline Regression - {sheetName}')
    # x labels should be integers only
    plt.xticks(ticks=[0, 1, 2, 3, 4, 5])
    plt.xlabel('Global Rating (1=Fail to 5=Excellent)')
    plt.ylabel(colScore)

    # show r2 value
    plt.text(0.05, 0.95, f"$R^2 = {r2:.3f}$", transform=plt.gca().transAxes, 
         fontsize=12, verticalalignment='top')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if savegraph:
        plt.savefig(f'{savefolder}/BLR {sheetName}.png')
    # plt.show()

    # ========== Summary Output ==========
    print("📊 BLR Model Summary")
    print(f"Cut Score (at Global = {passLevel}): {cutScore:.2f}")
    print(f"Cronbach's Alpha: {alpha:.3f}")
    print(f"R²: {r2:.3f} | Adjusted R²: {adj_r2:.3f}")
    print(f"Intercept: {intercept:.2f}")
    print(f"Slope (Inter-grade Discrimination): {interGradeDiscrimination:.2f}")
    print(f"✅ Pass Rate: {passRate:.1f}% | Failure Rate: {failureRate:.1f}% | Global Levels Used: {globalSpread} (Min: {globalCoverage['min']}, Max: {globalCoverage['max']})")
    print(f"Assessor Effect ANOVA F-Stat: {fStatass:.2f} | p-value: {pValass:.4f}")
    print(f"Assessor effect MLM: {assessorVar:.2f} | % Variance due to Assessor: {assessorPercent:.2f}%")
    print(f"ANOVA F-Stat: {f_stat:.2f} | p-value: {p_value:.4f}")
    # print(f"Item Code ANOVA F-Stat: {fStatItem:.2f} | p-value: {pValItem:.4f}")
    print("🔍 Normality Check (Shapiro-Wilk per Global Level):")
    for level, res in normalityResults.items():
        print(f"  Global={level} | n={res['n']} | W={res['stat']} | p={res['p']}")

    print(f"📏 Levene's Test for Equal Variances: stat={leveneStat:.2f}, p={leveneP:.4f}")
    if leveneP < 0.05:
        print("⚠️ Variances across global groups are unequal (violates ANOVA assumption).")

    
    statsDict = {
    "Station": sheetName,
    "CutScore": round(cutScore, 2),
    "CronbachAlpha": round(alpha, 3),
    "R2": round(r2, 3),
    "AdjR2": round(adj_r2, 3),
    "Intercept": round(intercept, 2),
    "Slope": round(interGradeDiscrimination, 2),
    "PassRate": round(passRate, 1),
    "FailureRate": round(failureRate, 1),
    "GlobalLevelsUsed": globalSpread,
    "GlobalMin": globalCoverage["min"],
    "GlobalMax": globalCoverage["max"],
    "AssessorANOVA_F": round(fStatass, 2),
    "AssessorANOVA_p": round(pValass, 4),
    "AssessorANOVA_Test": testType,
    "AssessorMLM_Var": round(assessorVar, 2),
    "AssessorMLM_%": round(assessorPercent, 2),
    "GlobalGroupANOVA_F": round(f_stat, 2),
    "GlobalGroupANOVA_p": round(p_value, 4),
    # "ItemCodeANOVA_F": round(fStatItem, 2),
    # "ItemCodeANOVA_p": round(pValItem, 4),
    # "ItemCodeANOVA_Test": testType,
    # "ItemCodeMLM_Var": round(itemCodeVar, 2),
    # "ItemCodeMLM_%": round(itemCodePercent, 2),
    "Levene_p": round(leveneP, 4),
    # "NormalityPerGroup": normalityResults,
    }

    if r2 >= 0.5:
        print("✅ Good correlation between global and checklist scores.")
    elif r2 < 0.3:
        print("⚠️ Weak correlation. Review station-level checklist design or examiner consistency.")
    return statsDict

infoDf = pd.DataFrame(columns=['Station', 'Cutoff'])
summaryDf = pd.DataFrame()
assessorDf = pd.DataFrame()
assessorcountinfo = pd.DataFrame()
passLevel = 2
for sheetName, df in sheetsDict.items():
    # if sheetName!='Station 15':
    #     continue
    print(f"\n=========Processing sheet: {sheetName}================")
    df['Item Codes'] = df['Item Codes'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    # take out the frist item code as string
    df['Item Codes'] = df['Item Codes'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)
    # valuecounts
    valuecountsitems = df['Item Codes'].value_counts()
    # # display(valuecountsitems)
    # if df['Item Codes'].nunique() > 1:
    #     # perform for each item subcode
    #     for itemCode in df['Item Codes'].unique():
    #         sub_df = df[df['Item Codes'] == itemCode]
    #         statsDict = brAnalysis(sub_df, passLevel=2, sheetName=itemCode)
    #         infoDf = pd.concat([infoDf, pd.DataFrame({'Station': [itemCode], 'Cutoff': [statsDict['CutScore']]})], ignore_index=True)
    #         summaryDf = pd.concat([summaryDf, pd.DataFrame([statsDict])], ignore_index=True)
    
    # # do for each rotation as well
    # rotations = df[colRotation].unique()
    # # sort them
    # rotations.sort()

    # if sheetName == 'Station 15':
    #     # only do for rotation 1 and 2
    #     rotations = [1, 2]
    #     sub_df = df[df[colRotation].isin(rotations)]
    #     statsDict = brAnalysis(sub_df, passLevel=2, sheetName=f'{sheetName} (Rotations 1 & 2)')
    #     summaryDf = pd.concat([summaryDf, pd.DataFrame([statsDict])], ignore_index=True)

    #     sub_df = df[~df[colRotation].isin(rotations)]
    #     statsDict = brAnalysis(sub_df, passLevel=2, sheetName=f'{sheetName} (Rotation 3)')
    #     summaryDf = pd.concat([summaryDf, pd.DataFrame([statsDict])], ignore_index=True)
    # else:
    #     rotations = sorted(rotations)
    #     for rotation in rotations:
    #         sub_df = df[df[colRotation] == rotation]
    #         statsDict = brAnalysis(sub_df, passLevel=2, sheetName=f'{sheetName} (Rotation {rotation})')
    #         # infoDf = pd.concat([infoDf, pd.DataFrame({'Station': [rotation], 'Cutoff': [statsDict['CutScore']]})], ignore_index=True)
    #         summaryDf = pd.concat([summaryDf, pd.DataFrame([statsDict])], ignore_index=True)

    # do for each assessor
    assessorcountinfo = pd.concat([assessorcountinfo, pd.DataFrame({'Station': [sheetName], 'Total Assessors': [df[colAssessorName].nunique()], 'Assessors with <5 records': [(df[colAssessorName].value_counts() < 5).sum()]})], ignore_index=True)
    assessors = df[colAssessorName].unique()
    if len(assessors) > 1:
        for assessor in assessors:
            sub_df = df[df[colAssessorName] == assessor]
            if len(sub_df) < 5:
                print(f"Skipping assessor {assessor} with only {len(sub_df)} records")
                continue
            statsDict = brAnalysis(sub_df, passLevel=passLevel, sheetName=f'{sheetName} ({assessor})', savegraph = False)
            # infoDf = pd.concat([infoDf, pd.DataFrame({'Station': [assessor], 'Cutoff': [statsDict['CutScore']]})], ignore_index=True)
            assessorDf = pd.concat([assessorDf, pd.DataFrame([statsDict])], ignore_index=True)

    statsDict = brAnalysis(df, passLevel=passLevel, sheetName=sheetName)
    infoDf = pd.concat([infoDf, pd.DataFrame({'Station': [sheetName], 'Cutoff': [statsDict['CutScore']]})], ignore_index=True)
    summaryDf = pd.concat([summaryDf, pd.DataFrame([statsDict])], ignore_index=True)
    # break

display(infoDf)
display(summaryDf)
display(assessorcountinfo)
filepath = f'BOH1 OSCE Stats ({today}) Level {passLevel}.xlsx'
if os.path.exists(os.path.join(folder, filepath)):
    with pd.ExcelWriter(os.path.join(folder, filepath), engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        infoDf.to_excel(writer, sheet_name='Cutoff Info', index=False)
        summaryDf.to_excel(writer, sheet_name='Summary', index=False)
        assessorDf.to_excel(writer, sheet_name='By Assessor', index=False)
else:
    with pd.ExcelWriter(os.path.join(folder, filepath), engine='openpyxl') as writer:
        infoDf.to_excel(writer, sheet_name='Cutoff Info', index=False)
        summaryDf.to_excel(writer, sheet_name='Summary', index=False)
        assessorDf.to_excel(writer, sheet_name='By Assessor', index=False)

### Pivot table

In [52]:
# cleanworkbookpath = '2025\\OSCE\\DDS2\\DDS2 OSCE unweighted (03-09-2025).xlsx'
# cleanworkbookpath = '2025\\OSCE\\DDS2\\SeparatedbyStation unweighted (15-10-2025) - Copy.xlsx'
cleanworkbookpath = '2025\\OSCE\\BOH2\\BOH2 OSCE Scores (09-12-2025).xlsx'
cleanworkbookpath = '2025\\OSCE\\BOH1\\BOH1 OSCE Scores (09-12-2025).xlsx'
cutoffpath = '2025\\OSCE\\BOH1\\BOH1 OSCE Stats (17-11-2025) Level 2.xlsx'

cutoffdf = pd.read_excel(cutoffpath, sheet_name='Cutoff Info')  
# there are two columns named 'Station'and Cutoff create a dictionary mapping station to cutoff
cutoffDict = pd.Series(cutoffdf.Cutoff.values,index=cutoffdf.Station).to_dict()
# values to int
cutoffDict = {int(k.split(' ')[1]): int(v) for k, v in cutoffDict.items()}
print(cutoffDict)
folder, file, ext = getFolderandFileName(cleanworkbookpath)
# iterate through the sheets in the workbook
# Read all sheets into dictionary of DataFrames
sheetsDict = pd.read_excel(cleanworkbookpath, sheet_name=None)
longDf = pd.DataFrame(columns=[colId, colIdanon, colStation, 'Total Score'])
longDfReadiness = pd.DataFrame(columns=[colId, colIdanon, colStation, colPractice])
for sheetName, df in sheetsDict.items():
    # Process each sheet's DataFrame (df) as needed
    # For example, you might want to extract specific columns or perform calculations
    # Here, we'll just append the 'Total Score' column to longDf
    longDf = pd.concat([longDf, df[[colId, colIdanon, colStation, 'Total Score']]], ignore_index=True)
    longDfReadiness = pd.concat([longDfReadiness, df[[colId, colIdanon, colStation, colPractice]]], ignore_index=True)

longDf['Total Score'] = pd.to_numeric(longDf['Total Score'], errors='coerce')
longDf['Total Score'] = longDf['Total Score'].astype('Int64') # allow NaN values
# display(longDf)

# pivot with station values as columns and the student id and anon ids as rows with score as values
pivotDf = longDf.pivot_table(index=[colId, colIdanon], columns=colStation, values='Total Score')
# change the type to int for stations
pivotDf = pivotDf.astype('Int64') # allow NaN values
# add a column with number of stations passed based on cutoffDict
def countPassedStations(row):
    passedCount = 0
    for station, cutoff in cutoffDict.items():
        if station in row and pd.notna(row[station]):
            print(row[station], cutoff)
            if row[station] >= cutoff:
                passedCount += 1
    return passedCount
pivotDf['Stations Passed'] = pivotDf.apply(countPassedStations, axis=1)
# display(pivotDf)
# display(infoDf)
pivotDf.to_excel(os.path.join(folder, f'pivot station unweighted ({today}).xlsx'), index=True)

{1: 48, 2: 53, 3: 46, 4: 42, 5: 59, 6: 53, 7: 66, 8: 57, 9: 53, 10: 43}
50 48
42 53
79 46
75 59
67 53
46 48
40 53
73 46
67 59
69 53
50 48
60 53
85 46
94 59
64 53
38 48
4 53
73 46
100 59
46 53
46 48
36 53
79 46
97 59
54 53
50 48
44 53
76 46
83 59
46 53


In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# Load the written Excel file
excelPath = os.path.join(folder, f'pivot station unweighted ({today}).xlsx')
wb = load_workbook(excelPath)
ws = wb.active  # Assuming single sheet

# Find starting column index for station scores
stationCols = list(cutoffDict.keys())
colOffset = 3  # Assuming first 2 columns are colId, colIdanon

# Create fill for failed cells
failFill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')

# Map station column headers to Excel columns
header = [cell.value for cell in ws[1]]
stationToCol = {station: header.index(station) + 1 for station in stationCols if station in header}

# Apply formatting
for row in range(2, ws.max_row + 1):  # Skip header row
    for station, cutoff in cutoffDict.items():
        colIdx = stationToCol.get(station)
        if not colIdx:
            continue
        cell = ws.cell(row=row, column=colIdx)
        try:
            score = int(cell.value)
            if score < cutoff:
                cell.fill = failFill
        except (TypeError, ValueError):
            continue

wb.save(excelPath)

In [ ]:
# Step 1: Create lookup for colPractice
readinessDict = {
    (row[colId], row[colStation]): row[colPractice]
    for _, row in longDfReadiness.iterrows()
}

# Step 2: Load Excel
excelPath = os.path.join(folder, f'pivot station unweighted ({today}).xlsx')
wb = load_workbook(excelPath)
ws = wb.active

# Step 3: Define fill style
purpleFill = PatternFill(start_color='D9D2E9', end_color='D9D2E9', fill_type='solid')  # light purple
lightblueFill = PatternFill(start_color='BDD7EE', end_color='BDD7EE', fill_type='solid')  # light blue
# Step 4: Get header for mapping station numbers to columns
header = [cell.value for cell in ws[1]]
stationCols = [c for c in header if isinstance(c, int)]  # Only station numbers

stationToColIdx = {station: header.index(station) + 1 for station in stationCols}

# Step 5: Apply purple fill where colPractice == 1
for row in range(2, ws.max_row + 1):
    studentId = ws.cell(row=row, column=1).value  # assumes colId is column 1
    for station in stationCols:
        colIdx = stationToColIdx[station]
        key = (studentId, station)
        if readinessDict.get(key) == 1:
            ws.cell(row=row, column=colIdx).fill = purpleFill
        # elif readinessDict.get(key) == 2:
        #     ws.cell(row=row, column=colIdx).fill = lightblueFill

wb.save(excelPath)

### PDF reports

In [ ]:
finalscorepath = '2025\\OSCE\\DDS2\\Final OSCE results updated.xlsx'
bystationpath = '2025\\OSCE\\DDS2\\Updated DDS2 OSCE data 3rd Sept 2025.xlsx'
reassessmentpath = '2025\\OSCE\\DDS2\\SeparatedbyStation unweighted (15-10-2025) - Copy.xlsx'
folder, file, ext = getFolderandFileName(finalscorepath)
finalDf = pd.read_excel(finalscorepath)
reassessPivot = pd.read_excel(finalscorepath, sheet_name='Reassessment')
cutoffs = pd.read_excel(finalscorepath, sheet_name='Cut scores')
byStationDf = pd.read_excel(bystationpath, sheet_name=None)
reassessmentDf = pd.read_excel(reassessmentpath, sheet_name=None)
# in byStationDf, set index to colId for each sheet
for sheetName, df in byStationDf.items():
    df.set_index(colId, inplace=True)

for sheetName, df in reassessmentDf.items():
    df.set_index(colId, inplace=True)
    # index to int
    # display(df.head())
    # df.index = df.index.astype('Int64')
finalDf.set_index(colId, inplace=True)
cutoffs.set_index('Station', inplace=True)
finalDf.index = finalDf.index.astype(int)
savefolder = f'{folder}/reports'
reassessPivot.set_index(colId, inplace=True)
reassessPivot.index = reassessPivot.index.astype(int)
print(f"Saving reports to {savefolder}")
os.makedirs(savefolder, exist_ok=True)
display(finalDf.head())
colFailed = '# Failed'
colPassed = '# Passed'
colReassessed = '# Reassessed'
colAverage = 'Updated to show failed 4 or more stations'
totStations = 11
colRotation = 'Rotation'
colComments = 'Proofread comments'
averageAll = int(finalDf[colAverage].mean().__round__(0))
print(f"Average score across all students: {averageAll}")
stationCols = finalDf.columns[3:3+totStations]
print(stationCols)
averageScores = {}
for station in stationCols:
    avg = int(finalDf[station].mean().__round__(0))
    averageScores[station] = avg
    print(f"Average score for {station}: {avg}")

scoreRanges =  {}
for station in stationCols:
    minScore = finalDf[station].min()
    maxScore = finalDf[station].max()
    scoreRanges[station] = (minScore, maxScore)
    print(f"Score range for {station}: {minScore} - {maxScore}")

stationNameDict  = {
    1: "Infection Control",
    2: "Communication with patient",
    4: "Communication with staff",
    5: "Medical Emergency",
    6: "Medical History",
    7: "Occlusion",
    9: "Radiography",
    10: "Caries risk",
    11: "LA",
    12: "Periodontology",
    14: "Conservative Dentistry",
    15: "Referral",
    16: "Informed Consent"
}

stationTypeDict = {
    1: "Observed Procedural",
    2: "Conversational - Simulation Patient",
    4: "Conversational - Simulation Clinical Educator",
    5: "Observed Procedural - Manikin",
    6: "Conversational - Simulation Patient",
    7: "Unobserved",
    9: "Observed Procedural - Manikin",
    10: "Conversational - Simulation Patient",
    11: "Observed Procedural - Manikin",
    12: "Conversational - Simulation Clinical Educator",
    14: "Conversational - Simulation Patient",
    15: "Unobserved",
    16: "Conversational - Simulation Patient"
}


In [ ]:
class BannerDrawer:
    def __init__(self, firstLine, secondLine, bannerWidth = None, bannerHeight=132, bgColor = '#010d44', topOffset=72, lineSpacing=36, leftMargin=36,
                 leftOffset = 36, topGap = 24):
        self.firstLine = firstLine
        self.secondLine = secondLine
        self.bannerWidth = bannerWidth
        self.bannerHeight = bannerHeight
        self.bgColor = bgColor
        self.topOffset = topOffset
        self.lineSpacing = lineSpacing
        self.leftMargin = leftMargin  # can replace variableUtils.leftMargin
        self.leftOffset = leftOffset
        self.topGap = topGap
        
    def __call__(self, canvas, doc):
        canvas.saveState()
        gapfill = 20

        pageWidth, pageHeight = doc.pagesize
        if self.bannerWidth is None:
            self.bannerWidth = pageWidth
        canvas.setFillColor(colors.HexColor(self.bgColor))
        # calculate x and y pos to make it centered
        x = (pageWidth - self.bannerWidth) / 2
        y = pageHeight - self.bannerHeight - self.topGap
        canvas.rect(x, y, self.bannerWidth, self.bannerHeight-self.topGap, fill=1, stroke=0)

        # Draw text
        canvas.setFillColor(colors.white)
        try:
            canvas.setFont("Calibri-Bold", 30)
        except:
            canvas.setFont("Helvetica-Bold", 30)
        canvas.drawString(self.leftMargin + self.leftOffset, pageHeight - self.topOffset - self.topGap - gapfill, self.firstLine)

        try:
            canvas.setFont("Calibri-Bold", 24)
        except:
            canvas.setFont("Helvetica-Bold", 24)
        canvas.drawString(self.leftMargin + self.leftOffset, pageHeight - self.topOffset - self.lineSpacing - self.topGap - gapfill, self.secondLine)

        canvas.restoreState()

In [ ]:
# Define styles
labelStyle = ParagraphStyle(name='LabelStyle', fontSize=8, alignment=TA_CENTER, textColor=colors.black, spaceAfter=2, leading=10)
valueStyle = ParagraphStyle(name='ValueStyle', fontSize=10, alignment=TA_CENTER, textColor=colors.black, spaceAfter=0)
# Light green inside table only
bgColor = '#dde8c8'
leftMargin = variableUtils.leftMargin*2
rightMargin = variableUtils.rightMargin*2
bannerWidth = pageSize[0] - (leftMargin + rightMargin)
bannerHeight = 100
topGap = 24
commonFactor = pageSize[0] - (leftMargin + rightMargin)
docElements = []

# Table cell creator
def boxCell(label, value):
    labelPara = Paragraph(f"<b>{label}</b>", labelStyle)
    valuePara = Paragraph(f"{value}", valueStyle)
    cellTable = Table([[labelPara, valuePara]], colWidths=[1.3 * inch, 1 * inch])
    cellTable.setStyle(TableStyle([
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOX', (1, 0), (1, 0), 0.5, colors.grey),
        ('BACKGROUND', (1, 0), (1, 0), colors.white),
        ('ALIGN', (1, 0), (1, 0), 'CENTER'),
        ('LEFTPADDING', (1, 0), (1, 0), 4),
        ('RIGHTPADDING', (1, 0), (1, 0), 4),
    ]))
    return cellTable

# Top-right cell: max score block
def maxScoreCell():
    maxScoreLabel = Paragraph("maximum station score=100", ParagraphStyle(name='MaxScoreStyle', fontSize=7, alignment=TA_RIGHT, textColor=colors.black))

    box = Table([[maxScoreLabel]], colWidths=[1.5* inch], rowHeights=[0.2 * inch])
    box.setStyle(TableStyle([
        # ('BOX', (0, 1), (0, 1), 0.5, colors.black),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('ALIGN', (0, 1), (0, 1), 'CENTER'),
    ]))
    return box

def drawBackground(canvas, doc, bgColor='#dce7c7'):
    canvas.saveState()
    # Convert hex to ReportLab color
    color = colors.HexColor(bgColor)
    canvas.setFillColor(color)

    # Draw rectangle covering the whole content area
    # pageSize is (width, height)
    pageWidth, pageHeight = doc.pagesize

    # Leave out the banner area at top
    left = doc.leftMargin
    bottom = doc.bottomMargin
    width = pageWidth - doc.leftMargin - doc.rightMargin
    height = pageHeight - bannerHeight - doc.bottomMargin - doc.topMargin

    canvas.rect(left, bottom, width, height, fill=1, stroke=0)
    canvas.restoreState()

def drawBackgroundOnLaterPages(canvas, doc):
    canvas.saveState()
    # Convert hex to ReportLab color
    color = colors.HexColor(bgColor)
    canvas.setFillColor(color)

    # Draw rectangle covering the whole content area
    # pageSize is (width, height)
    pageWidth, pageHeight = doc.pagesize

    # Leave out the banner area at top
    left = doc.leftMargin
    bottom = doc.bottomMargin
    width = pageWidth - doc.leftMargin - doc.rightMargin
    height = pageHeight - doc.bottomMargin - doc.topMargin

    canvas.rect(left, bottom, width, height, fill=1, stroke=0)
    canvas.restoreState()
# Data
def createTopEntry(examcolor, mark, averagemark, stationsPassed, stationsFailed, stationsbelowCutoff):
    data = [
        [boxCell("ROTATION:", examcolor), boxCell("MARK:", mark), boxCell("AVERAGE CLASS<br/>MARK:", averagemark), maxScoreCell()],
        [boxCell("# STATIONS<br/>PASSED:", stationsPassed), boxCell("# STATIONS<br/>FAILED:", stationsFailed), boxCell("# STATIONS < CUTOFF:", stationsbelowCutoff)]
    ]

    # Summary Table

    colWidths = [(commonFactor) *(5/6)/3]*3 + [(commonFactor) *(1/6)]
    print(colWidths)
    summaryBox = Table(data, colWidths=colWidths, rowHeights=[0.55 * inch]*2, hAlign='CENTER')
    summaryBox.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(bgColor)),
        # ('BOX', (0, 0), (-1, -1), 0.25, colors.black),
        # ('INNERGRID', (0, 0), (-1, -1), 0.25, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ]))


    # Add to doc
    topElements = [Spacer(1, bannerHeight-topMargin+topGap-5), summaryBox]
    return topElements

def createReassessmentTopEntry(examcolor, mark, stationsReassessed, stationsPassed, stationsFailed):
    data = [
        [boxCell("ROTATION:", examcolor), boxCell("MARK:", mark), boxCell("# STATIONS<br/>REASSESSED:", stationsReassessed), maxScoreCell()],
        [boxCell("# STATIONS<br/>PASSED:", stationsPassed), boxCell("# STATIONS<br/>FAILED:", stationsFailed)]
    ]

    # Summary Table

    colWidths = [(commonFactor) *(5/6)/3]*3 + [(commonFactor) *(1/6)]
    print(colWidths)
    summaryBox = Table(data, colWidths=colWidths, rowHeights=[0.55 * inch]*2, hAlign='CENTER')
    summaryBox.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(bgColor)),
        # ('BOX', (0, 0), (-1, -1), 0.25, colors.black),
        # ('INNERGRID', (0, 0), (-1, -1), 0.25, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ]))


    # Add to doc
    topElements = [summaryBox]
    return topElements

# Now create for each station
def createStationElement(stationName, stationScore, passFail, averagescore, scorerange, stationCutoff, speciality='', type='', domain='', comment = ''):
    elements = []
    headerelements = [[Paragraph(f"<b>STATION {stationName}</b>", ParagraphStyle(name='StationHeader', fontSize=12, alignment=TA_CENTER, textColor=colors.black, spaceAfter=12)),
                      Paragraph(f"<b>TOPIC:</b> {speciality}", ParagraphStyle(name='StationHeader', fontSize=10, alignment=TA_CENTER, textColor=colors.black, spaceAfter=12)),
                     Paragraph(f"<b>STATION TYPE:</b> {type}", ParagraphStyle(name='StationHeader', fontSize=10, alignment=TA_CENTER, textColor=colors.black, spaceAfter=12))]]
                    #   Paragraph(f"<b>DOMAIN:</b> {domain}", ParagraphStyle(name='StationHeader', fontSize=10, alignment=TA_CENTER, textColor=colors.black, spaceAfter=12))]]
    headerTable = Table(headerelements, colWidths=[(commonFactor) *(1/4)]*2+[(commonFactor) *(1/2)], hAlign='CENTER', rowHeights=[0.3 * inch])
    headerTable.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(bgColor)),
        # ('BOX', (0, 0), (-1, -1), 0.25, colors.black),
        # ('INNERGRID', (0, 0), (-1, -1), 0.25, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ]))
    # add a horizontal line
    # elements.append(Spacer(1, 6))
    line = HRFlowable(width="100%", thickness=4, lineCap='round', color='#010d44')
    data = [
        [boxCell("STATION OUTCOME:", passFail), boxCell("STATION SCORE:", f"{stationScore}"), boxCell("AVERAGE STATION SCORE:", f"{averagescore}")],
        [boxCell("STATION SCORE RANGE:", f"{scorerange}"), boxCell("STATION CUTOFF:", f"{stationCutoff}")]
    ]
    stationBox = Table(data, colWidths=[(commonFactor) *(1/3)]*3, rowHeights=[0.45 * inch]*2, hAlign='CENTER')
    stationBox.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor(bgColor)),
        # ('BOX', (0, 0), (-1, -1), 0.25, colors.black),
        # ('INNERGRID', (0, 0), (-1, -1), 0.25, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ]))
    # add for comment
    commentHeader = (Paragraph(f'<b>EXAMINER COMMENTS:</b>', ParagraphStyle(name='CommentStyle', fontSize=10, alignment=TA_LEFT, textColor=colors.black, spaceAfter=12, leftIndent=12, rightIndent=12)))
    
    # replace new lines in comment with <br/>
    if not pd.isna(comment):
        comment = comment.replace('\n', '<br/>')
    
    # add comment in a white box
    commentPara = (Paragraph(f'{comment}', ParagraphStyle(name='CommentStyle', fontSize=10, alignment=TA_LEFT, textColor=colors.black, spaceAfter=12, leftIndent=12, rightIndent=12)))
    commentBox = Table([[commentPara]], colWidths=[commonFactor*0.9], hAlign='LEFT')
    commentBox.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, -1), colors.white),
        ('BOX', (0, 0), (-1, -1), 0.5, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        # ('LEFTPADDING', (0, 0), (-1, -1), 6),
        # ('RIGHTPADDING', (0, 0), (-1, -1), 6),
        # ('TOPPADDING', (0, 0), (-1, -1), 6),
        # ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ]))
    elements.append(KeepTogether([line, headerTable, stationBox, Spacer(1, 6), commentHeader, commentBox]))
    return elements


In [ ]:
for idx, row in finalDf.iterrows():
    bgColor = '#dce7c7'
    studentID = idx
    name = row[colName]
    print(f"Processing student ID: {studentID} | Folder: {savefolder}")
    doc = SimpleDocTemplate(f"{savefolder}/{name} ({studentID}).pdf", pagesize=pageSize, rightMargin=rightMargin, leftMargin=leftMargin, 
                            topMargin=topMargin, bottomMargin=bottomMargin)
    elements = []
    examcolor = row[colRotation]
    rotation = row[colRotation]
    mark = row[colAverage].__round__(0)
    mark = int(mark)
    averagemark = averageAll
    stationsPassed = totStations - row[colFailed]
    stationsFailed = row[colFailed]
    stationsbelowCutoff = row[colFailed]  # assuming same as failed for now
    topEntry = createTopEntry(examcolor, mark, averagemark, stationsPassed, stationsFailed, stationsbelowCutoff)
    elements.extend(topEntry)
    # elements.append(Spacer(1, 24))
    if studentID in reassessPivot.index:
        reassessRow = reassessPivot.loc[studentID]
        examcolor = reassessRow[colRotation]
        mark = 50
        stationReassessed = reassessRow[colReassessed]
        stationsFailed = reassessRow[colFailed]
        stationsPassed = stationReassessed - stationsFailed
        topEntryRe = createReassessmentTopEntry(examcolor, mark, stationReassessed, stationsPassed, stationsFailed)
        lineT = HRFlowable(width="100%", thickness=4, lineCap='round', color='#010d44')
        elements.append(lineT)
        elements.append(Paragraph(f"<b>REASSESSMENT RESULTS</b>", ParagraphStyle(name='ReassessHeader', fontSize=14, alignment=TA_CENTER, textColor='#010d44', spaceAfter=12)))
        elements.extend(topEntryRe)
        elements.append(Spacer(1, 24))

    
    # for each station, create station element
    print(stationCols)
    for station in stationCols:
        stationScore = row[station]
        averagescore = averageScores[station]
        scorerangetext = f"{scoreRanges[station][0]} - {scoreRanges[station][1]}"
        stationCutoff = cutoffs.loc[f'Station {station} (Rotation {rotation})', 'CutScore']
        passFail = "PASS" if stationScore >= stationCutoff else "FAIL"
        
        
        # add examiner comments from byStationDf if available
        key = f'Station {station}'
        comment = byStationDf[key].loc[studentID, colComments]
        # check if comment is not string
        if not isinstance(comment, str):
            print(f"Comment for student ID {studentID} at {key} is not a string. Using empty comment. {comment}")
            comment = ''
        # if pd.isna(comment) or comment.strip() == '':
        #     comment = byStationDf[key].loc[studentID, 'Assessor Feedback'] # fallback to Assessor Feedback
        stationElement = createStationElement(station, stationScore, passFail, averagescore, scorerangetext, stationCutoff, comment=comment, speciality = stationNameDict[station], type=stationTypeDict[station])
        elements.extend(stationElement)
        elements.append(Spacer(1, 12))

        #if station is fail add assessment info from reassessmentDf
        if passFail == "PASS":
            continue
        reassessDf = reassessmentDf[f'Station {station}']
        # check if studentID is in reassessDf index
        if studentID not in reassessDf.index:
            print(f"Student ID {studentID} not found in reassessment data for Station {station}. Skipping reassessment info.")
            continue
        print(f'Reassessment for student ID {studentID} at Station {station}')
        stationScore = reassessDf.loc[studentID, 'Total Score']
        passFail = "PASS" if stationScore >= stationCutoff else "FAIL"
        comment = reassessDf.get(colComments, reassessDf.get('Proofread Comments')).loc[studentID]
        if not isinstance(comment, str):
            print(f"Reassessment comment for student ID {studentID} at Station {station} is not a string. Using empty comment. {comment}")
            comment = ''
        stationElement = createStationElement(f'{station}-Reassess', stationScore, passFail, averagescore, scorerangetext, stationCutoff, comment=comment, speciality = stationNameDict[station], type=stationTypeDict[station])
        elements.extend(stationElement)
        elements.append(Spacer(1, 12))
        
    
    banner = BannerDrawer('DDS2 OSCE Feedback', name, bannerWidth, bannerHeight, topOffset=36, leftMargin=leftMargin, topGap=topGap)
    doc.build(elements, onFirstPage=lambda c, d: (banner(c, d), drawBackground(c, d, bgColor)), onLaterPages=lambda c, d:  drawBackgroundOnLaterPages(c, d))
    # break


## CAF

In [ ]:
workbookpath = '2025\DDS2\\all_data_combined_v2.xlsx'
# workbookpath = '2025/BOH1/all_data_combined_v2.xlsx'
# workbookpath = '2025/DDS1/all_data_combined_v2.xlsx'
# workbookpath = '2025\\BOH2\\all_data_combined_v2.xlsx'
# workbookpath = '2025\\Viva\\BOH2\\all_data_combined_v2.xlsx'
# workbookpath = '2025\\Viva\\DDS3\\all_data_combined_v2.xlsx'
# workbookpath = '2025\\mini CEX\\DDS2\\all_data_combined_v2.xlsx'
# workbookpath = '2025\\BOH2\\Smile Squad\\Processed_CAFs.xlsx'
folder, file, ext = getFolderandFileName(workbookpath)
df = pd.read_excel(workbookpath)
# define variable names
colId = 'Student ID'
colName = 'Student Name'
colDate = 'Date'
colCI = 'CI'
colTS = 'TS'
colES = 'ES'
colCS = 'CS'
colPS = 'PS'
colScale = 'Section Rating'
colPractice = 'Practice Readiness'
colAssessorName = 'Assessor Name'
colAssessorFeedback = 'Assessor Feedback'
colStudentFeedback = 'Student Feedback'
colPatientComplexity = 'Patient Complexity'
# display(df.head())
# drop na Student ID
df.rename(columns={'student_number':colId, 'student_name': colName, 'datetime': colDate,
                   'time_mgmt': colTS, 'entrustment': colES, 'communication': colCS, 'professionalism': colPS, 
                   'scale-section-rating': colScale, 'scale-practice-readiness': colPractice,
                   'clinical_incident': 'CI', 'patient_complexity': 'Patient Complexity', 'assessor_name': colAssessorName,
                   'assessor_feedback': colAssessorFeedback,
                   'student_feedback': colStudentFeedback}, inplace=True, errors='ignore')
display(df.head())
df.dropna(subset=[colId], inplace=True)
df[colId] = df[colId].astype(int)
# df[colTS] = df[colTS].astype('Int64')
# df[colES] = df[colES].astype('Int64')
# df[colCS] = df[colCS].astype('Int64')
# if colPS in df.columns:
#     df[colPS] = df[colPS].astype('Int64', errors='ignore')  # some values are strings
# df[colScale] = df[colScale].astype('Int64')
# df[colPractice] = df[colPractice].astype('Int64')

headingStyleLarge = ParagraphStyle('Heading1', parent=styles['Heading1'], fontSize=72, alignment=1)  # Centered

# df[colPatientComplexity] = df[colPatientComplexity].fillna('')
# df[colPatientComplexity] = df[colPatientComplexity].astype(str)
# df[colCI] = df[colCI].fillna('')
commonList = [
    'assessment_id', colName, colId, colAssessorName, colDate,
    'cohort', 'subject', 'type', 'student_submitted', 'assessor_submitted',
    colTS, colES, colCS, colPS, colScale, colPractice,
    colAssessorFeedback, colCI, colPatientComplexity, colStudentFeedback
]
colClinicType = 'clinic'
beforeCols = ['assessment_id', 'form_name', colName, colId,  colDate, colAssessorName, 'cohort', 'subject', 'type', colPatientComplexity,
              colClinicType]
rubricCols = [colTS, colES, colCS, colPS, colScale, colPractice]
feedbackCols = [colCI, colAssessorFeedback, colStudentFeedback ]   

replaceNotReviwedwithNo = False
display(df.head())

# get weights of marking checklist and rubrics
weightPS = 0.00
weightTS = 0.05
weightES = 0.10
weightCS = 0.05
weightMC = round(1 - weightPS - weightTS - weightES - weightCS, 3)
print(f"Weight of PS: {weightPS}, TS: {weightTS}, ES: {weightES}, CS: {weightCS}, MC: {weightMC}")

In [ ]:
def extractCodes(supervisorDataStr):
    try:
        data = json.loads(supervisorDataStr)
        codes = data.keys()
        return sorted(codes)
    except Exception as e:
        print(f"Error extracting codes: {e}")
        return []
    
# Expand the JSON with scores having two levels of keys
def expandJson(row):
    jsonDict = json.loads(row)
    flatDict = {}
    for outerKey, innerDict in jsonDict.items():
        for innerKey, value in innerDict.items():
            flatDict[f'{outerKey}_{innerKey}'] = value
    return pd.Series(flatDict)

# get row wise scores for each item code
def getRowWiseScores(df):
    df['supervisor_data'] = df['supervisor_data'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

    scoresList = []
    notinabel = []

    for idx, row in df.iterrows():    
        print(f"Row: {idx}")
        jsonData = row['supervisor_data']
        if not isinstance(jsonData, dict):
            print(f"Row {idx} has no supervisor_data or it is not a dictionary\n{jsonData}")
            scoresList.append(json.dumps({}))
            continue
        scoreDict = {}
        # print(f'Item Codes: {itemList}')
        for item, mcDict in jsonData.items():
            # print(f'Item Code: {item}')
            # replace Not Reviewed and Not Assessed with No in mcDict values
            try:
                mcDict = {k: ('NA' if v in ['Not Reviewed', 'Not Assessed'] else v) for k, v in mcDict.items()} 
            except AttributeError as e:
                print(f"AttributeError: mcDict for item {item} in row {idx} is not a dictionary\n{mcDict}\n{e}")
                notinabel.append((item, 'entire mcDict'))
                return
                # continue
            if replaceNotReviwedwithNo:
                mcDict = {k: ('No' if v =='NA' else v) for k, v in mcDict.items()}

            scoreDict[item] = {}
            cutoff = 0
            weightedCutoff = 0
            weightedSum = 0
            denominator = 0
            nYes = 0
            nNo = 0
            notNADict = {k: v for k, v in mcDict.items() if v != 'NA'}
            nNA  = len(mcDict) - len(notNADict)
            for key, value in notNADict.items(): # key is MC1, MC2 etc and value is Yes/No
                if value not in ['Yes', 'No']:
                    print(f"Unexpected value {value} for item {item} and key {key}")
                    continue
                if value == 'Yes':
                    nYes += 1
                elif value == 'No':
                    nNo += 1
                try:
                    cutoff += ebelWeights[item].loc[key, 'ebelWeight']
                    weightedCutoff += ebelWeights[item].loc[key, 'ebelWeight'] * ebelWeights[item].loc[key, 'scoreWeight']
                    denominator += ebelWeights[item].loc[key, 'scoreWeight']
                    if value == 'Yes':
                        weightedSum += ebelWeights[item].loc[key, 'scoreWeight']
                    elif value == 'No':
                        weightedSum += 0
                except KeyError as e:
                    print(f"KeyError: {item} {key} not found in ebelWeights for row {idx}\n{e}")
                    notinabel.append((item, key))
                    continue
            
            scoreDict[item]['Yes'] = nYes
            scoreDict[item]['No'] = nNo
            scoreDict[item]['NA'] = nNA
            scoreDict[item]['% Yes'] = round((nYes / (nYes + nNo) * 100), 2) if (nYes + nNo) > 0 else np.nan

            if not notNADict:
                print(f'Item Code: {item} has no data')
                scoreDict[item] = scoreDict[item] | {'Cutoff': np.nan, 'Weighted Cutoff': np.nan, 'Weighted Sum': np.nan, 'Denominator': np.nan,
                                   'Weighted Score': np.nan, 'Total Score': np.nan}
                continue
            
            try:
                cutoff = cutoff / len(notNADict)
            except ZeroDivisionError as e:
                print(f'ZeroDivisionError for item {item} with data: {notNADict}\n{e}')
                cutoff = np.nan
            
            try:
                weightedCutoff = weightedCutoff / denominator
                weightedScore = round((weightedSum / denominator * 100), 2)
            except ZeroDivisionError as e:
                print(f'ZeroDivisionError for item {item} with data: {notNADict}\n{e}')
                weightedCutoff = np.nan
                weightedScore = np.nan

                
            # CS TS ES PS  all should become 0 if no value is present
            # row[colTS] = row[colTS] if pd.notna(row[colTS]) else 0
            # row[colES] = row[colES] if pd.notna(row[colES]) else 0
            # row[colCS] = row[colCS] if pd.notna(row[colCS]) else 0
            # row[colPS] = row[colPS] if pd.notna(row[colPS]) else 0

            # totalScore = weightMC * weightedScore + (weightTS * row[colTS]/4 + weightES * row[colES]/4  + weightCS * row[colCS]/2)*100
            # totalCutoff = weightMC * weightedCutoff + (weightTS * 0.5 + weightES * 0.5 + weightPS * 0.5 + weightCS * 0.5)*100
            # get dtype of each item in the above expression

            # print(f'Cutoff: {cutoff}, Weighted Cutoff: {weightedCutoff}, Weighted Sum: {weightedSum}, Denominator: {denominator}')
            try:
                scoreDict[item] = scoreDict[item] | {'Cutoff': round(cutoff,2), 'Weighted Cutoff': round(weightedCutoff,2), 'Weighted Sum': int(weightedSum), 'Denominator': int(denominator),
                                                    'Weighted Score': weightedScore,}# 'Total Score': round(totalScore, 2), 'Total Cutoff': round(totalCutoff, 2)}
            except TypeError:
                print(f"Error in item {item} with data: {scoreDict[item]}, student data: {row[colId]}")
                print(cutoff, weightedCutoff, weightedSum, denominator, weightedScore,)# totalScore)
        # pprint(scoreDict)        
        scoresList.append(json.dumps(scoreDict))
    df['Scores'] = scoresList
    df['supervisor_data'] = df['supervisor_data'].apply(lambda x: json.dumps(x, indent=2, ensure_ascii=False))
    return df, notinabel

In [ ]:
df_ = df.copy()
df_['Item Codes'] = df_['supervisor_data'].apply(extractCodes)
_, notinabel = getRowWiseScores(df_)
df_.to_excel(f'{folder}\\all data scores_v2.xlsx')
if notinabel:
    items = set()
    print("The following item/key pairs were not found in ebelWeights:")
    for item, key in set(notinabel):
        print(f"Item: {item}, Key: {key}")
        items.add(item)
    print(f"Affected item codes: {items}")

In [ ]:
# combine the scores with the v1 scores
df_v1 = pd.read_excel(os.path.join(folder, 'all data scores.xlsx'))
df_v2 = pd.read_excel(os.path.join(folder, 'all data scores_v2.xlsx'))
print(f'V1 columns: {df_v1.columns.tolist()}')
print(f'V2 columns: {df_v2.columns.tolist()}')
print(set(df_v1.columns) ^ set(df_v2.columns))  # symmetric difference
# merge them vertically
df_combined = pd.concat([df_v2, df_v1], ignore_index=True)
display(df_combined.head())
df_combined.to_excel(os.path.join(folder, 'all data scores_combined.xlsx'), index=False)

### Attendance check

In [ ]:
datesToMatchDt = {
    datetime(2025, 7, 28): ['532', '532'],
    datetime(2025, 8, 4): ['525'],
    datetime(2025, 8, 18): ['414', '524'],
    datetime(2025, 8, 25): ['414', '586'],
    datetime(2025, 9, 1): ['586'],
    datetime(2025, 9, 8): ['414', '579', '386', '656'],
    datetime(2025, 9, 15): ['161', '013', '121', '013']
}

In [ ]:

df[colDate] = pd.to_datetime(df[colDate], errors='coerce').dt.date
df = df[df[colDate].isin([d.date() for d in datesToMatchDt.keys()])]
print(f"Records after date filtering: {len(df)}")
pecCodes = ['Consent', 'Record_keeping', 'infection_control', 'positioning']
df['Item Codes'] = df['supervisor_data'].apply(extractCodes)
# remove the pec codes from item codes
df['Item Codes'] = df['Item Codes'].apply(lambda x: [code for code in x if code not in pecCodes])
display(df.head())

# see all the clinics
clinicTypes = df[colClinicType].unique()
print(f"Clinic types: {clinicTypes}")
# filter Paeds
df = df[df[colClinicType] == 'Paeds']
print(f"Records after clinic filtering: {len(df)}")
# display(df.head())
allStudentIds = getStudentList('2025\RE_ Student List.xlsx', cohort='DDS2')
print(f"Total students in cohort: {len(allStudentIds)}")
# attendance check for each date
attendanceRecords = []
# attendance + code check
missingRecords = []

# see submiited by assessor counts
counts = df['submitted_by_assessor'].value_counts()
studentcounts = df['submitted_by_student'].value_counts()
print(f"Submitted by student counts:\n{studentcounts}")
print(f"Submitted by assessor counts:\n{counts}")
# filter out where student submitted but assessor did not
dfstudentsubmitted = df[(df['submitted_by_assessor'] == False) & (df['submitted_by_student'] == True)]
dfstudentsubmitted = dfstudentsubmitted[[colId, colName, colDate, colAssessorName, 'submitted_by_student', 'submitted_by_assessor', 'Item Codes', 'supervisor_data']]
dfstudentsubmitted.to_excel(os.path.join(folder, f'Student Submitted Assessor Not({today}).xlsx'), index=False)
# take only those submitted by assessor
df = df[df['submitted_by_assessor'] == True]
print(f"Records after filtering for submitted by assessor: {len(df)}")
def normalizeCode(code):
    # Extract leading 3 digits if present, else return the whole string
    m = re.match(r'^(\d{3})', str(code))
    return m.group(1) if m else str(code).strip()

for student in allStudentIds:
    student = int(student)
    studentRecords = df[df[colId] == student]
    name = studentRecords[colName].iloc[0] if not studentRecords.empty else 'Unknown'
    if studentRecords.empty:    
        print(f"Student {student} - {name} has no records")
        for date, requiredCodes in datesToMatchDt.items():
            dateOnly = date.date()
            missingRecords.append({
                colId: student,
                'Student Name': name,
                'Date': dateOnly,
                'MissingCodes': requiredCodes
            })
    for date, requiredCodes in datesToMatchDt.items():
        dateOnly = date.date()
        codesDone = studentRecords.loc[studentRecords[colDate] == dateOnly, 'Item Codes']
        codesDone = [normalizeCode(c) for sub in codesDone for c in sub]  # flatten

        # find missing codes
        missingCodes = [code for code in requiredCodes if code not in codesDone]
        missingCodes = ', '.join(missingCodes) if missingCodes else ''
        # find extra codes
        extraCodes = [code for code in codesDone if code not in requiredCodes]
        extraCodes = ', '.join(extraCodes) if extraCodes else ''
        if missingCodes:
            missingRecords.append({
                colId: student,
                'Student Name': name,
                'Date': dateOnly,
                'MissingCodes': missingCodes,
                'ExtraCodes': extraCodes
            })

# final database of missing codes
missingDf = pd.DataFrame(missingRecords)
display(missingDf.head())
print(f"Total missing records: {len(missingDf)}")
missingDf.to_excel(os.path.join(folder, f'Attendance and Code Check Paeds ({today}).xlsx'), index=False)

### Patient Age and item codes

In [ ]:
path1 = '2025\\BOH2\\age codes.xlsx'
path2 = '2025\\BOH3+DDS4\\Session 7\\BOH3 age codes.xlsx'
path3 = '2025\\BOH2\\Smile Squad\\Processed_Age_ItemCodes.xlsx'
folder = '2025\\BOH2'
df1 = pd.read_excel(path1)
df1['Cohort'] = 'BOH2'
df2 = pd.read_excel(path2)
df2['Cohort'] = 'BOH3'
df3 = pd.read_excel(path3)
df3['Cohort'] = 'BOH2'
# display(df1.head())
# display(df2.head())
colAge = 'Age'

validAges = df2[df2[colAge] != 1][colAge]

# Get probability distribution of valid ages
ageDist = validAges.value_counts(normalize=True)
# display(ageDist)
# Number of rows with Age = 1
nWrong = (df2[colAge] == 1).sum()

# Randomly sample replacement ages
replacementAges = np.random.choice(
    ageDist.index, size=nWrong, replace=True, p=ageDist.values
)

# print(replacementAges)
# df2.loc[df2[colAge] == 1, colAge] = replacementAges
dfage1 = df2[df2[colAge] == 1]
dfage1.loc[:, colAge] = replacementAges
df2 = df2[df2[colAge] != 1] # removing rows with age 1


df = pd.concat([df1, df2, df3], ignore_index=True)
df['Code'] = df['Code'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
dfage1['Code'] = dfage1['Code'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df[colAge] = pd.to_numeric(df[colAge], errors='coerce')
df = df.dropna(subset=[colAge])
df[colAge] = df[colAge].astype(int)
df = df[df[colAge] > 0]
df.to_excel(f'{folder}/age codes combined ({today}).xlsx', index=False)
# create age distribution

def plotages(df, title, colAge = 'Age'):
    ages = df[colAge].tolist()
    ages = [int(age) for age in ages if age >= 0 and age < 18] # only ages 0-17
    total = len(ages)
    # save age list to npy file
    # print(ages)
    fig, ax = plt.subplots(figsize=(figSize[0]*1.2, figSize[1]*0.8))
    sns.histplot(ages, bins=[i for i in range(0, 18, 1)], ax=ax)
    plt.xticks(ticks=[i for i in range(0, 18, 1)])
    # plt.yticks(ticks=[i for i in range(0, max(4, int(len(ages)/5), 10), 100)])
    # show total number of ages
    # total = len(df)
    plt.text(0.90, 0.90, f'Total: {total}', horizontalalignment='center', verticalalignment='center', transform=ax.transAxes)
    plt.title(title)

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xlabel('Age')
    plt.savefig(f'{folder}/graphs/{title} ({today}).png')
    return fig



# age groups 0-5 6-11 12-17 and 18+
def ageGroup(age):
    if age <=5:
        return '0-5'
    elif age <=11:
        return '6-11'
    elif age <=17:
        return '12-17'
    elif age>=18:
        return '18+'
    else:
        return pd.NA

def plotAgeGroupCodes(df, agegroup, cohort):
    if agegroup is  None or agegroup == 'Under 18':
        subdf = df
    else:
        subdf = df[df['Age Group'] == agegroup]
    totalRecords = len(subdf)
    if totalRecords == 0:
        print(f'Age Group: {agegroup} has no records, skipping')
        return None
    display(subdf.head())
    print(f'Age Group: {agegroup}, n={len(subdf)}')
    allCodes = [code for sublist in subdf['Code'] for code in sublist if isinstance(sublist, list)]
    print(f'Total codes before cleaning: {len(allCodes)} [unique: {len(set(allCodes))}]')
    codeFixes = {
    '112': '012',
    '144': '141',
    '021': '221',
    '211': '221',
    '212': '221',
    '002': '012',
    '331': '311',
    '027': '927',
    '970': '927',
    '122': '121',
    '001': '011',
    '079': '970'
    }
    # Codes to exclude
    excludeCodes = ['741', '776', '316', '091', '766', '770', '351']

    # Cleaned codes list
    allCodes = [
    codeFixes.get(code, code)   # replace if in fixes
    for code in allCodes
    if code not in excludeCodes # filter unwanted
    ]
    codeCounts = pd.Series(allCodes).value_counts()
    display(codeCounts.head(10))
    # remove dropitems from codeCounts
    dropitems = ['infection_control', 'positioning', 'Record_keeping', 'Consent']
    codeCounts = codeCounts[~codeCounts.index.isin(dropitems)]
    # remove less than 6 counts
    # codeCounts = codeCounts[codeCounts.values >= 6]
    # take top 20 codes
    codeCounts = codeCounts.head(35)
    # remove codes with more than 10 characters in name
    codeCounts = codeCounts[codeCounts.index.str.len() <= 10]
    # display(codeCounts.head(10))
    # plot bar chart of codeCounts
    # --- Professional bar chart ---
    fig, ax = plt.subplots(figsize=(figSize[0]*1.6, figSize[1]*0.6))
    bars = ax.bar(codeCounts.index.astype(str), codeCounts.values)
    
    # Titles and labels
    ax.set_title(f'Item Code Distribution (Top 35) — Age Group ({agegroup})', fontsize=14)
    ax.set_xlabel('Item Codes', fontsize=12)
    ax.set_ylabel('Counts', fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # y ticks to integers only
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    # Rotate x labels if too many codes
    plt.xticks(rotation=45, ha='right')
    # y limit atleast 4 to show bars properly
    ax.set_ylim(0, max(4, codeCounts.values.max()*1.1))
    
    # Add numbers on top of bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, 
                f'{int(height)}', ha='center', va='bottom', fontsize=10)
    # ax.text(0.95, 0.95, f'Total Records: {totalRecords}', transform=ax.transAxes, 
    #      fontsize=10, verticalalignment='bottom', horizontalalignment='right', color='black', alpha=1)
    plt.tight_layout()
    # plt.show()
    # save the figure
    os.makedirs(f'{folder}/graphs', exist_ok=True)
    fig.savefig(f'{folder}/graphs/Item Code Distribution - Age Group {agegroup} {cohort} ({today}).png')
    return fig

df['Age Group'] = df['Age'].apply(ageGroup)

# plot for each age group
agegroups = df['Age Group'].unique()
agegroups = np.sort(df['Age Group'].unique())

def plotAll(df, cohort) :
    print(f'Plotting for cohort: {cohort}')
    for agegroup in agegroups:
        plotAgeGroupCodes(df, agegroup, cohort)

    plotages(df, f'Age Distribution of Patients in {cohort}')

# plotAll(df, 'BOH')
# dfboh2 = df[df['Cohort'] == 'BOH2']
# dfboh3 = df[df['Cohort'] == 'BOH3']
# plotAll(dfboh2, 'BOH2')
# plotAll(dfboh3, 'BOH3')
# dfage1['Age Group'] = dfage1[colAge].apply(ageGroup)
# plotages(dfage1, f'Age Distribution of Patients in BOH3 (with correction)')
# plotAll(dfage1, 'BOH3 (with correction)')
dfunder18 = df[df[colAge] < 18]
plotAll(dfunder18, 'Under 18')
plotAgeGroupCodes(dfunder18, 'Under 18', 'BOH')

## Separated Item Df

In [ ]:
def flattenJson(jsonData):
    flatDict = {}
    for outerKey, innerDict in jsonData.items():
        if isinstance(innerDict, dict):
            for innerKey, value in innerDict.items():
                flatDict[f'{outerKey}_{innerKey}'] = value
        else:
            flatDict[outerKey] = innerDict
    return flatDict

pecCodes = ['Consent', 'Record_keeping', 'infection_control', 'positioning']
def getSeparatedItemDf(df, dftype_='by item'):
    dfDict = {}
    # check if item codes is a string and convert to list
    df['Item Codes'] = df['Item Codes'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    itemCodes = df['Item Codes'].sum()

    itemCodes = list(set(itemCodes))
    for item in itemCodes:
        if item in pecCodes:
            continue
        # print(f'Item Code: {item}')
        # itemDf.set_index('assessment_id', inplace=True)
        itemDf = df[df['Item Codes'].apply(lambda x: item in x)] # select rows where item is in Item Codes
        if isinstance(itemDf['supervisor_data'].iloc[0], str):
            # print("Converting supervisor_data to JSON")
            # convert to json
            itemDf['supervisor_data'] = itemDf['supervisor_data'].apply(lambda x: json.loads(x))
            itemDf['supervisor_data'] = itemDf['supervisor_data'].apply(flattenJson)
        supervisorDataDf= itemDf['supervisor_data'].apply(pd.Series)
        if replaceNotReviwedwithNo:
            supervisorDataDf.replace({'Not Reviewed': 'No'}, inplace=True)
        supervisorDataDf.replace({'Not Reviewed': 'NA', 'Not Assessed': 'NA'}, inplace=True)
        # display(itemDf.head())
        # display(supervisorDataDf.head())
        # only take columns have item code in them
        # display(supervisorDataDf.head())
        validCols = [col for col in supervisorDataDf.columns if f'{item}_MC' in col]
        supervisorDataDf = supervisorDataDf[validCols]
        # rename columns to remove assessor_ text
        supervisorDataDf.columns = [col.split('assessor_')[-1] for col in supervisorDataDf.columns]

        # Step 1: Convert Yes/No to 1/0
        supervisordataDfBinary = supervisorDataDf.replace({'Yes': 1, 'No': 0, 'NA': np.nan})
        # display(supervisordataDfBinary.head())
        # Step 2: Calculate column scores
        columnScores = supervisordataDfBinary.sum(axis=0)

        # Step 3: Sort columns based on score (descending)
        sortedColumns = columnScores.sort_values(ascending=False).index
        sortedColumns = supervisorDataDf.columns # unsort the columns
        # Step 4: Reorder dataframe columns
        sortedSupervisorDataDf = supervisorDataDf[sortedColumns]
        sortedSupervisorDataDf = sortedSupervisorDataDf.replace({'Yes': 1, 'No': 0})
        # remove item_ from column names
        sortedSupervisorDataDf.columns = [col.split(f'{item}_')[-1] for col in sortedSupervisorDataDf.columns]
        # display(sortedSupervisorDataDf.head())
        # display(supervisorDataDf.head())
        thiscommonList = [col for col in itemDf.columns if col in commonList]
        thisbeforeCols = [col for col in beforeCols if col in itemDf.columns]
        thisrubricCols = [col for col in rubricCols if col in itemDf.columns]
        combinedDf= pd.concat([itemDf[thisbeforeCols], sortedSupervisorDataDf, itemDf[thisrubricCols]], axis=1)
        # display(combinedDf.head())
        expandedScores = itemDf['Scores'].apply(expandJson)
        validCols2 = [col for col in expandedScores.columns if item in col]
        expandedScores = expandedScores[validCols2]
        # rename columns to remove item_ text
        expandedScores.columns = [col.split(f'{item}_')[-1] for col in expandedScores.columns]
        # display(expandedScores.head())
        combinedDf = pd.concat([combinedDf, expandedScores], axis=1)
        thisfeedbackCols = [col for col in feedbackCols if col in itemDf.columns]
        combinedDf = pd.concat([combinedDf, itemDf[thisfeedbackCols]], axis=1)
        if dftype_ == 'by item':
            display(combinedDf.head())
            dfDict[item] = combinedDf
        
        elif dftype_ == 'by date':
            for dateVal, subDf in itemDf.groupby(colDate):
                dateStr = pd.to_datetime(dateVal).strftime('%Y-%m-%d')
                combinedDf = pd.concat([
                    subDf[thisbeforeCols],
                    sortedSupervisorDataDf.loc[subDf.index],
                    subDf[thisrubricCols],
                    expandedScores.loc[subDf.index],
                    subDf[thisfeedbackCols],
                ], axis=1)

                dfDict[f'{item} ({dateStr})'] = combinedDf
        
        elif dftype_ == 'by week':
            # group by week and combine
            itemDf[colDate] = pd.to_datetime(itemDf[colDate], format='mixed')
            itemDf['weekStart'] = itemDf[colDate].dt.to_period('W-MON').apply(lambda r: r.start_time)

            for weekVal, subDf in itemDf.groupby('weekStart'):
                dateStr = pd.to_datetime(weekVal).strftime('%Y-%m-%d')
                combinedDf = pd.concat([
                    subDf[thisbeforeCols],
                    sortedSupervisorDataDf.loc[subDf.index],
                    subDf[thisrubricCols],
                    expandedScores.loc[subDf.index],
                    subDf[thisfeedbackCols],
                ], axis=1)

                dfDict[f'{item} ({dateStr})'] = combinedDf
        
    
    return dfDict



In [ ]:
# df_ = pd.read_excel(f'{folder}\\all data scores_v2.xlsx')
df_ = pd.read_excel(f'2025\\BOH2\\Smile Squad\\Processed_CAFs.xlsx')
display(df_.head())

In [ ]:
type_ = 'Simulation'
type_ = 'Clinic'
type_ = None
if type_ is not None:
    df_ = df_[df_['type'] == type_]  # filter for type

# get separated item df by item
dfDictItem = getSeparatedItemDf(df_, dftype_='by item')
# save the dfs to same workbook with different sheet names
filepath = f'{folder}\\SeparatedbyItems {type_} ({today}).xlsx' if type_ else f'{folder}\\SeparatedbyItems ({today}).xlsx'
if os.path.exists(filepath):
    os.remove(filepath)
# sort the keys of dfDict by date and item code
dfDict = {k: v for k, v in sorted(dfDictItem.items(), key=lambda item: (item[0]))}
display(dfDict.keys())
with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
    for item, itemDf in dfDict.items():
        print(f'Saving {item} to {filepath}')   
        itemDf.to_excel(writer, sheet_name=item.replace('/', '_') , index=False)
        print(f'Saved {item} to {filepath}')

# get separated item df by date
dfDictItemandDate = getSeparatedItemDf(df_, dftype_='by date')
filepath = f'{folder}\\SeparatedbyItemsandDate {type_} ({today}).xlsx' if type_ else f'{folder}\\SeparatedbyItemsandDate ({today}).xlsx'
if os.path.exists(filepath):
    os.remove(filepath)
dfDictItemandDate = {k: v for k, v in sorted(dfDictItemandDate.items(), key=lambda item: (pd.to_datetime(item[1][colDate].iloc[0]), item[0]))}
display(dfDictItemandDate.keys())
with pd.ExcelWriter(filepath, engine='openpyxl') as writer: 
    for item, itemDf in dfDictItemandDate.items():
        print(f'Saving {item} to {filepath}')   
        itemDf.to_excel(writer, sheet_name=item.replace('/', '_'), index=False)
        print(f'Saved {item} to {filepath}')

# get separated item df by week
dfDictItemandWeek = getSeparatedItemDf(df_, dftype_='by week')
filepath = f'{folder}\\SeparatedbyItemsandWeek {type_} ({today}).xlsx' if type_ else f'{folder}\\SeparatedbyItemsandWeek ({today}).xlsx'
if os.path.exists(filepath):
    os.remove(filepath)
dfDictItemandWeek = {k: v for k, v in sorted(dfDictItemandWeek.items(), key=lambda item: (pd.to_datetime(item[1][colDate].iloc[0]), item[0]))}
display(dfDictItemandWeek.keys())
with pd.ExcelWriter(filepath, engine='openpyxl') as writer: 
    for item, itemDf in dfDictItemandWeek.items():
        print(f'Saving {item} to {filepath}')   
        itemDf.to_excel(writer, sheet_name=item.replace('/', '_'), index=False)
        print(f'Saved {item} to {filepath}')

## Student Reports

In [ ]:
workbookpath = f'2025\\BOH2\\all data scores_v2.xlsx'
# workbookpath = f'2025\\BOH1\\all data scores_v2.xlsx'
# workbookpath = f'2025\\DDS2\\all data scores_v2.xlsx'

folder, file, ext = getFolderandFileName(workbookpath)
df = pd.read_excel(workbookpath)
pecItems = ['Consent', 'Record_keeping', 'infection_control', 'positioning']
rubricCols = [colTS, colES, colCS, colPS]
otherCols = [
    "form_name", "assessment_id", "Student Name", "Student ID", "Date", "cohort",
    "subject", "type", "completed", "role", "clinic", "patient_age", "patient_drn", "Assessor Name",
    "patient_details", "patient_interpreter", "Patient Complexity",
    # "assessor_submitted", "student_submitted",
    "fta_substitute", "teeth_quadrant_details", "Student Feedback", "Assessor Feedback",
]
otherfullnamedict = {'student_name': 'Student Name', 'assessor_name': 'Assessor Name', 'date': 'Date', 'cohort': 'Cohort', 'subject': 'Subject', 'type': 'Type',
                     'time_mgmt': 'Time Management', 'communication': 'Communication', 'professionalism': 'Professionalism', 'entrustment': 'Entrustment',
                     'student_feedback': 'Student Feedback', 'assessor_feedback': 'Assessor Feedback', 'clinical_incident': 'Clinical Incident',
                     'patient_complexity': 'Complexity', 'Consent': 'Consent'}
rubricTexts = {colTS: {"name": "Time Management Scale", "fields": {"1": "Level 1: Work not completed in allocated timeframe.", "2": "Level 2: Reaches a step (or point) in the procedures where it would be safe for the patient to leave the clinic.", "3": "Level 3: Completes simulation or clinical procedures in allocated timeframe.", "4": "Level 4: Manages time to complete simulation or clinical procedures, dental records (clinic only) and assessment forms within the allocated timeframe."}}, 
               colES: {"name": "Entrustment Scale", "fields": {"1": "Level 1: Student cannot be trusted to perform this task.", "2": "Level 2: Student can be trusted to perform this task with direct supervision.", "3": "Level 3: Student can be trusted to perform this task with indirect supervision.", "4": "Level 4: Student can be trusted to perform this task independently."}}, 
               colCS: {"name": "Communication Scale", "fields": {"1": "Level 1: Communicates with the patient and assessor", "2": "Level 2: Matches verbal and non-verbal communication strategies to the message being communicated. Uses active listening."}}, 
               colPS: {"name": "Professionalism Scale", "fields": {"1": "Level 1: Attends the clinic or simulation clinic", "2": "Level 2: Presents as a professional (e.g. clothing, hair, on time, etc.)"}}, 
               "patient_complexity": {"name": "Patient Complexity", "fields": {"complex": "Complex", "non-complex": "Non-complex"}}}

df[colDate] = pd.to_datetime(df[colDate], format='mixed')
# sort by date
df = df.sort_values(by=colDate, ascending=True)
# select between specified dates
# startDate = datetime(2025, 10, 13)
# endDate = datetime(2025, 10, 22)
# df = df[(df[colDate] >= startDate) & (df[colDate] <= endDate)]

In [ ]:
display(df[df[colId] == 1088656])

In [ ]:
def loadJson(element):
    if isinstance(element, dict):
        return element
    elif isinstance(element, str):  
        try:
            return json.loads(element)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            return {}
    else:
        print(f"Unexpected type for JSON loading: {type(element)}")
        return {}
    
def createRowReport(row, idx):
    elements = []
    row[colDate] = row[colDate].strftime('%Y-%m-%d')
    itemDescJson = loadJson(row['checklists'])
    itemDescAdditional = loadJson(row['additional_checklists'])
    elements.append(Paragraph(f'Form {idx+1}: {row[colDate]}', subheadingStyle))
    elements.append(Spacer(1, 12))
    # covnert item codes to list if not already
    if isinstance(row['Item Codes'], str):
        row['Item Codes'] = ast.literal_eval(row['Item Codes'])
    if len(row['Item Codes']) > 0:
        elements.append(Spacer(1, 12))
        elements.append(Paragraph('Items Performed', smallsubsubheadingStyleL))
        elements.append(Spacer(1, 12))
        elements.append(Paragraph(', '.join(row['Item Codes']), tableTextStyleL))
    elements.append(Spacer(1, 12))
    
    for col in otherCols:
        
        if col not in row or pd.isna(row[col]) or row[col] == 'nan' or row[col] == '':
            continue
        elements.append(Paragraph(f'{otherfullnamedict[col] if col in otherfullnamedict.keys() else col}', smallsubsubheadingStyleL))
        elements.append(Spacer(1, 4))
        # if type is not string, convert to string
        if not isinstance(row[col], str):
            row[col] = str(row[col])
        text = row[col].strip()#.replace('\n', '<br/>')
        # remove _x000D_
        text = re.sub(r'_x000D_', '', text)
        # remove non-ascii characters
        text = re.sub(r'[^\x00-\x7F]+', '', text)
        text = text.replace('&', '&amp;')
        text = text.replace('<', '&lt;')
        text = text.replace('>', '&gt;')
        text = text.replace('\n', '<br/>')  # replace newlines with <br/>
        elements.append(Paragraph(f'{text}', tableTextStyleLSmall))
        elements.append(Spacer(1, 8))

    # add rubric table 
    rubricDf = pd.DataFrame(columns = ['Rubric', 'Score', 'Description'])
    for col in rubricCols:
        if col in row and not pd.isna(row[col]) and row[col] != 'nan':
            score = int(row[col])
            rubricDf = pd.concat([rubricDf, pd.DataFrame([{
                'Rubric': rubricTexts[col]['name'],
                'Score': score,
                'Description': rubricTexts[col]['fields'].get(str(score), 'No description available')
            }])], ignore_index=True)

    if not rubricDf.empty:
        rubricTable = createTable(rubricDf, 'Rubric Scores', colRatio=[2, 1, 5], customTextCols=[0, 1, 2], tableTextStyle=tableTextStyleSmall,
                                    topPadding=6, bottomPadding=6)
        elements.append(rubricTable)
        elements.append(Spacer(1, 12))

    # add mc table
    studentData = loadJson(row['student_data'])
    supervisorData = loadJson(row['supervisor_data'])

    print(f'Form: {row["form_name"]}')
    for item in row['Item Codes']:
        if item in pecItems:
            continue
        mcDf = pd.DataFrame(columns=['Marking Checklist', 'Description', 'Student', 'Supervisor'])
        itemName = itemDescJson.get(item, {}).get('name', item)
        print(f'Student Data for item {item}: {studentData}\n Supervisor Data: {supervisorData}')
        mcKeys = itemDescJson.get(item, {}).get('fields', [])
        if not mcKeys:
            print(f"No fields found for item {item} in row {idx}")
            continue
        for mcKey in mcKeys:
            studentrecord = studentData.get(item, {}).get(mcKey, '')
            studentrecord = 'NA' if studentrecord in ['Not Reviewed', 'Not Assessed'] else studentrecord
            supervisorrecord = supervisorData.get(item, {}).get(mcKey, '')
            supervisorrecord = 'NA' if supervisorrecord in ['Not Reviewed', 'Not Assessed'] else supervisorrecord
            mcdesc = itemDescJson.get(item, {}).get('fields', {}).get(mcKey, '')
            # print(f'STudent record: {studentrecord}')
            # print(f'Supervisor record: {supervisorrecord}')
            record = {
                'Marking Checklist': mcKey,
                'Description': mcdesc,
                'Student': studentrecord,
                'Supervisor': supervisorrecord
            }
            print(f"Record for item {item}, MC Key {mcKey}: {record}")
            mcDf = pd.concat([mcDf, pd.DataFrame([record])], ignore_index=True)
        if mcDf.empty:
            print(f"No data found for item {item} in row {idx}")
            continue
        mcTable = createTable(mcDf, f'Evaluation for {item}: {itemName}', colRatio = [1.4, 5, 1, 1], customTextCols=[0, 1], cellHighlight=True, 
                              tableTextStyle=tableTextStyleSmall, topPadding=6, bottomPadding=6)
        elements.append(mcTable)
        elements.append(Spacer(1, 12))
    
    # add addtional checklists in the pecCols
    mcDf = pd.DataFrame(columns=['Additional Checklist', 'Description', 'Supervisor'])
    for item in row['Item Codes']:
        if item not in pecItems:
            continue
        if item not in itemDescAdditional:
            print(f"No additional checklist found for item {item} in row {idx}")
            continue

        itemName = itemDescAdditional.get(item, {}).get('name', item)
        print(f'Student Data for item {item}: {studentData}\n Supervisor Data: {supervisorData}')
        mcKeys = itemDescAdditional.get(item, {}).get('fields', [])
        if not mcKeys:
            print(f"No fields found for item {item} in row {idx}")
            continue
        for mcKey in mcKeys:
            studentrecord = studentData.get(item, {}).get(mcKey, '')
            studentrecord = 'NA' if studentrecord in ['Not Reviewed', 'Not Assessed'] else studentrecord
            supervisorrecord = supervisorData.get(item, {}).get(mcKey, '')
            supervisorrecord = 'NA' if supervisorrecord in ['Not Reviewed', 'Not Assessed'] else supervisorrecord
            mcdesc = itemDescAdditional.get(item, {}).get('fields', {}).get(mcKey, '')
            record = {
                'Additional Checklist': f'{item}_{mcKey}',
                'Description': mcdesc,
                'Supervisor': supervisorrecord
            }
            print(f"Record for item {item}, MC Key {mcKey}: {record}")
            mcDf = pd.concat([mcDf, pd.DataFrame([record])], ignore_index=True)
    
    if not mcDf.empty:    
        mcTable = createTable(mcDf, f'Evaluation for Additional Items', colRatio = [2, 5, 1], customTextCols=[0, 1], cellHighlight=True, 
                                tableTextStyle=tableTextStyleSmall, topPadding=6, bottomPadding=6)
        elements.append(mcTable)
        elements.append(Spacer(1, 12))
    
    # # add scores table
    # scoresJson = json.loads(row['Scores']) if isinstance(row['Scores'], str) else row['Scores']
    # if not scoresJson:
    #     print(f"No scores found for row {idx}")
    #     return elements
    return elements

def createWholeReport(df, studentId):
    elements = []

    elements.append(Spacer(1, 72))
    studentDf = df[df[colId] == studentId]
    
    studentName = studentDf[colName].iloc[0]
    savefolder = f'{folder}\\Student Reports V3'
    os.makedirs(savefolder, exist_ok=True)
    doc = SimpleDocTemplate(f'{savefolder}\\{studentId} ({studentName}).pdf', pagesize= pageSize, leftMargin = leftMargin,
                            rightMargin = rightMargin, topMargin = topMargin, bottomMargin = bottomMargin)
    
    for i, (idx, row) in enumerate(studentDf.iterrows()):
        print(f'Creating report for {studentName} ({studentId}) - Form {i+1}')
        elements.extend(createRowReport(row, i))
        elements.append(PageBreak())
    
    doc.build(elements, onFirstPage=getBannerDrawer('Student Form Report', f'{studentName} ({studentId})'),)

studentIds = df[colId].unique()
# create anon student ids and assessor names

print(f"Total unique student IDs: {len(studentIds)}")
for studentId in studentIds:
    # if studentId!= 1088656:
    #     continue
    print(f'Creating report for student ID: {studentId}')
    createWholeReport(df, studentId)
    # break

